In [ ]:
import os
import gc
import re
import json
import time
import math
import random
import warnings
import unicodedata
import traceback
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import transformers
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoModelForMultipleChoice,
    AutoConfig,
    get_cosine_schedule_with_warmup,
    set_seed as hf_set_seed
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

import psutil
import GPUtil

warnings.filterwarnings("ignore")

print("Torch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.5.1+cu121
Transformers version: 4.46.3
CUDA available: True
GPU name: Tesla T4


In [ ]:
GLOBAL_SEED = 42

def set_all_seeds(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as e:
        print("Deterministic algorithms partial support:", e)
    hf_set_seed(seed)
    print(f"All seeds set to {seed}")

set_all_seeds(GLOBAL_SEED)

All seeds set to 42


In [ ]:
@dataclass
class HardwareConfig:
    device: str = "cpu"
    device_name: str = "cpu"
    total_vram_gb: float = 0.0
    has_tensor_cores: bool = False
    amp_dtype: torch.dtype = torch.float32
    use_amp: bool = False
    compute_capability: Tuple[int, int] = (0, 0)

def detect_hardware() -> HardwareConfig:
    cfg = HardwareConfig()
    if torch.cuda.is_available():
        cfg.device = "cuda"
        cfg.device_name = torch.cuda.get_device_name(0)
        props = torch.cuda.get_device_properties(0)
        cfg.total_vram_gb = props.total_memory / (1024 ** 3)
        cfg.compute_capability = (props.major, props.minor)
        cfg.has_tensor_cores = props.major >= 7
        if cfg.has_tensor_cores:
            cfg.use_amp = True
            cfg.amp_dtype = torch.bfloat16 if props.major >= 8 else torch.float16
        else:
            cfg.use_amp = True
            cfg.amp_dtype = torch.float16
    else:
        cfg.device = "cpu"
        cfg.device_name = "cpu"
        cfg.total_vram_gb = 0.0
        cfg.use_amp = False
        cfg.amp_dtype = torch.float32
    return cfg

HW = detect_hardware()
print("Device:", HW.device)
print("Device name:", HW.device_name)
print("Total VRAM (GB):", round(HW.total_vram_gb, 2))
print("Compute capability:", HW.compute_capability)
print("Has tensor cores:", HW.has_tensor_cores)
print("AMP dtype selected:", HW.amp_dtype)
DEVICE = torch.device(HW.device)

Device: cuda
Device name: Tesla T4
Total VRAM (GB): 14.84
Compute capability: (7, 5)
Has tensor cores: True
AMP dtype selected: torch.float16


In [ ]:
def benchmark_amp(hw_cfg: HardwareConfig, hidden_size=768, seq_len=128, batch_size=16, steps=20):
    if hw_cfg.device != "cuda":
        return hw_cfg

    model = nn.Sequential(
        nn.Linear(hidden_size, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, hidden_size),
        nn.ReLU(),
        nn.Linear(hidden_size, hidden_size)
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    dummy_input = torch.randn(batch_size, seq_len, hidden_size, device=DEVICE)
    dummy_target = torch.randn(batch_size, seq_len, hidden_size, device=DEVICE)

    torch.cuda.synchronize()
    start = time.time()
    for _ in range(steps):
        optimizer.zero_grad()
        out = model(dummy_input)
        loss = F.mse_loss(out, dummy_target)
        loss.backward()
        optimizer.step()
    torch.cuda.synchronize()
    fp32_time = time.time() - start

    scaler = GradScaler(enabled=True)
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(steps):
        optimizer.zero_grad()
        with autocast(dtype=hw_cfg.amp_dtype):
            out = model(dummy_input)
            loss = F.mse_loss(out, dummy_target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    torch.cuda.synchronize()
    amp_time = time.time() - start

    del model, optimizer, dummy_input, dummy_target
    gc.collect()
    torch.cuda.empty_cache()

    print(f"FP32 time for {steps} steps: {fp32_time:.4f}s")
    print(f"AMP time for {steps} steps: {amp_time:.4f}s")

    if amp_time >= fp32_time * 0.98:
        print("AMP did not provide meaningful speedup on this GPU. Disabling AMP for speed, memory savings still applied selectively.")
        hw_cfg.use_amp = False
    else:
        speedup = fp32_time / amp_time
        print(f"AMP provided {speedup:.2f}x speedup. Keeping AMP enabled.")
        hw_cfg.use_amp = True

    return hw_cfg

HW = benchmark_amp(HW)
print("Final AMP decision:", HW.use_amp, "| dtype:", HW.amp_dtype)

FP32 time for 20 steps: 1.5088s
AMP time for 20 steps: 0.1777s
AMP provided 8.49x speedup. Keeping AMP enabled.
Final AMP decision: True | dtype: torch.float16


In [ ]:
BASE_DIR = Path(r"C:/Users/VGST/Desktop/CryptoQA")

TASK1_DIR = BASE_DIR / "Task 1"
TASK1_TEST_DIR = TASK1_DIR / "task1_test"

TASK2_DIR = BASE_DIR / "Task 2"
TASK2_TEST_DIR = TASK2_DIR / "task2_test_dataset"

OUTPUT_DIR = BASE_DIR / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
LOG_DIR = OUTPUT_DIR / "logs"
EDA_DIR = OUTPUT_DIR / "eda"
METRICS_DIR = OUTPUT_DIR / "metrics"
PLOTS_DIR = OUTPUT_DIR / "plots"

TASK1_PATHS = {
    "reddit_train": TASK1_DIR / "CRYPTO-REDDIT-OPINION-TRAIN.csv",
    "twitter_train": TASK1_DIR / "CRYPTO-TWITTER-OPINION-TRAIN.csv",
    "youtube_train": TASK1_DIR / "CRYPTO-YOUTUBE-OPINION-TRAIN.csv",
    "reddit_test": TASK1_TEST_DIR / "CRYPTO_REDDIT_TEST.csv",
    "twitter_test": TASK1_TEST_DIR / "CRYPTO_TWITTER_TEST.csv",
    "youtube_test": TASK1_TEST_DIR / "CRYPTO_YOUTUBE_TEST.csv"
}

TASK2_PATHS = {
    "mcq_train": TASK2_DIR / "mcq_train.jsonl",
    "mcq_val": TASK2_DIR / "mcq_val.jsonl",
    "qa_train": TASK2_DIR / "qa_train.jsonl",
    "qa_val": TASK2_DIR / "qa_val.jsonl",
    "qna_train_csv": TASK2_DIR / "QnA-TRAIN.csv",
    "videos": TASK2_DIR / "videos.jsonl",
    "qna_test": TASK2_TEST_DIR / "CRYPTO_QnA_TEST.csv",
    "mcq_test": TASK2_TEST_DIR / "task2_mcq_test.jsonl",
    "qa_test": TASK2_TEST_DIR / "task2_qa_test.jsonl"
}

SUBMISSION_DIRS_TASK1 = {
    1: OUTPUT_DIR / "Task1" / "Submission_1",
    2: OUTPUT_DIR / "Task1" / "Submission_2",
    3: OUTPUT_DIR / "Task1" / "Submission_3"
}

SUBMISSION_DIRS_TASK2 = {
    1: OUTPUT_DIR / "Task2" / "Submission_1",
    2: OUTPUT_DIR / "Task2" / "Submission_2",
    3: OUTPUT_DIR / "Task2" / "Submission_3"
}

ALL_DIRS = [
    OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, EDA_DIR, METRICS_DIR, PLOTS_DIR
] + list(SUBMISSION_DIRS_TASK1.values()) + list(SUBMISSION_DIRS_TASK2.values())

for d in ALL_DIRS:
    d.mkdir(parents=True, exist_ok=True)

print("Directory structure created")
for d in ALL_DIRS:
    print(str(d))

Directory structure created
C:\Users\VGST\Desktop\CryptoQA\outputs
C:\Users\VGST\Desktop\CryptoQA\outputs\checkpoints
C:\Users\VGST\Desktop\CryptoQA\outputs\logs
C:\Users\VGST\Desktop\CryptoQA\outputs\eda
C:\Users\VGST\Desktop\CryptoQA\outputs\metrics
C:\Users\VGST\Desktop\CryptoQA\outputs\plots
C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_1
C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_2
C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_3
C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_1
C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_2
C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_3


In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def get_gpu_memory_stats():
    if not torch.cuda.is_available():
        return {"allocated_gb": 0.0, "reserved_gb": 0.0, "free_gb": 0.0, "total_gb": 0.0}
    allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
    reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)
    total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    free = total - reserved
    return {
        "allocated_gb": round(allocated, 3),
        "reserved_gb": round(reserved, 3),
        "free_gb": round(free, 3),
        "total_gb": round(total, 3)
    }

def log_gpu_usage(tag=""):
    stats = get_gpu_memory_stats()
    print(f"[{tag}] Allocated: {stats['allocated_gb']}GB | Reserved: {stats['reserved_gb']}GB | Free: {stats['free_gb']}GB | Total: {stats['total_gb']}GB")
    return stats

def delete_model_safely(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    clear_memory()

class OOMSafeRunner:
    def __init__(self, min_batch_size=1, reduction_factor=2):
        self.min_batch_size = min_batch_size
        self.reduction_factor = reduction_factor

    def run(self, fn, batch_size, *args, **kwargs):
        current_batch_size = batch_size
        while current_batch_size >= self.min_batch_size:
            try:
                result = fn(current_batch_size, *args, **kwargs)
                return result, current_batch_size
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"OOM at batch size {current_batch_size}. Reducing batch size.")
                    clear_memory()
                    current_batch_size = max(self.min_batch_size, current_batch_size // self.reduction_factor)
                    if current_batch_size < self.min_batch_size:
                        print("Minimum batch size reached. Raising exception.")
                        raise
                else:
                    raise
        raise RuntimeError("Could not find a working batch size without OOM")

def safe_execute(fn, *args, default_return=None, log_prefix="", **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"[{log_prefix}] Execution failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        clear_memory()
        return default_return

OOM_RUNNER = OOMSafeRunner()
log_gpu_usage("initial")

[initial] Allocated: 0.062GB | Reserved: 0.123GB | Free: 14.719GB | Total: 14.842GB


{'allocated_gb': 0.062,
 'reserved_gb': 0.123,
 'free_gb': 14.719,
 'total_gb': 14.842}

In [ ]:
@dataclass
class TrainingConfig:
    model_name: str = ""
    max_seq_length: int = 128
    per_device_batch_size: int = 16
    gradient_accumulation_steps: int = 2
    effective_batch_size: int = field(init=False)
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    num_epochs: int = 4
    dropout: float = 0.1
    label_smoothing: float = 0.05
    max_grad_norm: float = 1.0
    early_stopping_patience: int = 2
    use_amp: bool = HW.use_amp
    amp_dtype: torch.dtype = HW.amp_dtype
    num_workers: int = 0
    pin_memory: bool = True
    persistent_workers: bool = False
    prefetch_factor: int = None
    use_gradient_checkpointing: bool = True

    def __post_init__(self):
        self.effective_batch_size = self.per_device_batch_size * self.gradient_accumulation_steps

DEFAULT_CONFIG = TrainingConfig()
print("Effective batch size target:", DEFAULT_CONFIG.effective_batch_size)
print("AMP enabled:", DEFAULT_CONFIG.use_amp, "| dtype:", DEFAULT_CONFIG.amp_dtype)
print("num_workers:", DEFAULT_CONFIG.num_workers, "| persistent_workers:", DEFAULT_CONFIG.persistent_workers)

Effective batch size target: 32
AMP enabled: True | dtype: torch.float16
num_workers: 0 | persistent_workers: False


In [ ]:
def fix_unicode(text):
    if not isinstance(text, str):
        return ""
    try:
        import ftfy
        text = ftfy.fix_text(text)
    except Exception:
        pass
    text = unicodedata.normalize("NFKC", text)
    return text

def normalize_whitespace(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def clean_text(text):
    text = fix_unicode(text)
    text = normalize_whitespace(text)
    return text

def is_empty_text(text):
    if text is None:
        return True
    if not isinstance(text, str):
        return True
    return len(text.strip()) == 0

def detect_language_simple(text):
    try:
        from langdetect import detect
        if is_empty_text(text) or len(text.strip()) < 3:
            return "unknown"
        return detect(text)
    except Exception:
        return "unknown"

def sanitize_name(name):
    return name.replace("/", "_").replace("-", "_")

In [ ]:
LEVEL1_CODE_MAP = {0: "NOISE", 1: "OBJECTIVE", 2: "SUBJECTIVE"}
LEVEL2_CODE_MAP = {0: "NEUTRAL", 1: "NEGATIVE", 2: "POSITIVE"}
LEVEL3_CODE_MAP = {0: "NEUTRAL SENTIMENT", 1: "QUESTION", 2: "ADVERTISEMENT", 3: "MISCELLANEOUS"}

TASK1_TRAIN_SOURCE_CONFIG = {
    "reddit": {"text_cols": ["MAIN"], "level1": "Level 1", "level2": "Level 2", "level3": "Level 3", "id_col": None},
    "twitter": {"text_cols": ["Tweet"], "level1": "Level 1", "level2": "Level 2", "level3": "Level 3", "id_col": None},
    "youtube": {"text_cols": ["comment"], "level1": "Level 1", "level2": "Level 2", "level3": "Level 3", "id_col": "comment_id"}
}

TASK1_TEST_SOURCE_CONFIG = {
    "reddit": {"text_cols": ["MAIN"], "level1": None, "level2": None, "level3": None, "id_col": None},
    "twitter": {"text_cols": ["Text"], "level1": None, "level2": None, "level3": None, "id_col": "Unnamed: 0"},
    "youtube": {"text_cols": ["MAIN"], "level1": None, "level2": None, "level3": None, "id_col": "comment_id"}
}

def code_to_label(value, code_map):
    if pd.isna(value):
        return np.nan
    try:
        code = int(float(value))
    except (ValueError, TypeError):
        return np.nan
    return code_map.get(code, np.nan)

print("Task 1 schema configs ready")
print("Train config:", TASK1_TRAIN_SOURCE_CONFIG)
print("Test config:", TASK1_TEST_SOURCE_CONFIG)

Task 1 schema configs ready
Train config: {'reddit': {'text_cols': ['MAIN'], 'level1': 'Level 1', 'level2': 'Level 2', 'level3': 'Level 3', 'id_col': None}, 'twitter': {'text_cols': ['Tweet'], 'level1': 'Level 1', 'level2': 'Level 2', 'level3': 'Level 3', 'id_col': None}, 'youtube': {'text_cols': ['comment'], 'level1': 'Level 1', 'level2': 'Level 2', 'level3': 'Level 3', 'id_col': 'comment_id'}}
Test config: {'reddit': {'text_cols': ['MAIN'], 'level1': None, 'level2': None, 'level3': None, 'id_col': None}, 'twitter': {'text_cols': ['Text'], 'level1': None, 'level2': None, 'level3': None, 'id_col': 'Unnamed: 0'}, 'youtube': {'text_cols': ['MAIN'], 'level1': None, 'level2': None, 'level3': None, 'id_col': 'comment_id'}}


In [ ]:
def load_and_clean_task1_csv(filepath, source_name, source_config):
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return None, None

    df = None
    for encoding in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(filepath, encoding=encoding)
            break
        except Exception:
            continue

    if df is None:
        print(f"Failed to load {filepath} with any known encoding")
        return None, None

    df.columns = [str(c).strip() for c in df.columns]
    original_len = len(df)

    cfg = source_config[source_name]
    available_text_cols = [c for c in cfg["text_cols"] if c in df.columns]
    if not available_text_cols:
        print(f"[{source_name}] None of expected text columns {cfg['text_cols']} found in {list(df.columns)}")
        return None, None

    def build_text(row):
        parts = []
        for c in available_text_cols:
            val = row[c]
            if pd.notna(val) and str(val).strip() != "":
                parts.append(str(val))
        return " ".join(parts)

    df["text_raw"] = df.apply(build_text, axis=1)

    level1_col = cfg["level1"] if cfg["level1"] and cfg["level1"] in df.columns else None
    level2_col = cfg["level2"] if cfg["level2"] and cfg["level2"] in df.columns else None
    level3_col = cfg["level3"] if cfg["level3"] and cfg["level3"] in df.columns else None

    df["level1"] = df[level1_col].apply(lambda x: code_to_label(x, LEVEL1_CODE_MAP)) if level1_col else np.nan
    df["level2"] = df[level2_col].apply(lambda x: code_to_label(x, LEVEL2_CODE_MAP)) if level2_col else np.nan
    df["level3"] = df[level3_col].apply(lambda x: code_to_label(x, LEVEL3_CODE_MAP)) if level3_col else np.nan

    if cfg["id_col"] and cfg["id_col"] in df.columns:
        df["record_id"] = df[cfg["id_col"]]
    else:
        df["record_id"] = range(len(df))

    df["text"] = df["text_raw"].apply(clean_text)
    df["is_empty"] = df["text"].apply(is_empty_text)

    missing_report = {
        "text_missing": int(df["text_raw"].isna().sum()),
        "level1_missing": int(df["level1"].isna().sum()) if level1_col else None,
        "level2_missing": int(df["level2"].isna().sum()) if level2_col else None,
        "level3_missing": int(df["level3"].isna().sum()) if level3_col else None,
        "empty_after_cleaning": int(df["is_empty"].sum())
    }

    df = df[~df["is_empty"]].reset_index(drop=True)

    duplicate_count_before = int(df.duplicated(subset=["text"]).sum())
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

    df["source"] = source_name
    df["char_length"] = df["text"].apply(len)
    df["word_count"] = df["text"].apply(lambda x: len(x.split()))

    stats = {
        "source": source_name,
        "original_rows": original_len,
        "final_rows": len(df),
        "duplicates_removed": duplicate_count_before,
        "missing_report": missing_report,
        "text_cols_used": available_text_cols,
        "level_cols_used": [level1_col, level2_col, level3_col],
        "id_col_used": cfg["id_col"]
    }

    print(f"[{source_name}] Original: {original_len} -> Final: {len(df)} rows")
    print(f"[{source_name}] Duplicates removed: {duplicate_count_before}")
    print(f"[{source_name}] Missing report: {missing_report}")
    if level1_col:
        print(f"[{source_name}] level1 value counts:\n{df['level1'].value_counts(dropna=False)}")

    return df, stats

In [ ]:
task1_dataframes = {}
task1_load_stats = {}

for key in ["reddit_train", "twitter_train", "youtube_train"]:
    source_name = key.replace("_train", "")
    df, stats = safe_execute(
        load_and_clean_task1_csv,
        TASK1_PATHS[key],
        source_name,
        TASK1_TRAIN_SOURCE_CONFIG,
        default_return=(None, None),
        log_prefix=f"load_{key}"
    )
    if df is not None:
        task1_dataframes[source_name] = df
        task1_load_stats[source_name] = stats

print("Loaded Task 1 training sources:", list(task1_dataframes.keys()))
for source, df in task1_dataframes.items():
    print(f"{source}: {df.shape}")

[reddit] Original: 5000 -> Final: 4962 rows
[reddit] Duplicates removed: 38
[reddit] Missing report: {'text_missing': 0, 'level1_missing': 0, 'level2_missing': 1148, 'level3_missing': 1817, 'empty_after_cleaning': 0}
[reddit] level1 value counts:
level1
SUBJECTIVE    3839
NOISE          622
OBJECTIVE      501
Name: count, dtype: int64
[twitter] Original: 4288 -> Final: 3759 rows
[twitter] Duplicates removed: 527
[twitter] Missing report: {'text_missing': 0, 'level1_missing': 0, 'level2_missing': 3038, 'level3_missing': 3384, 'empty_after_cleaning': 2}
[twitter] level1 value counts:
level1
OBJECTIVE     1474
NOISE         1167
SUBJECTIVE    1118
Name: count, dtype: int64
[youtube] Original: 5000 -> Final: 4763 rows
[youtube] Duplicates removed: 237
[youtube] Missing report: {'text_missing': 0, 'level1_missing': 0, 'level2_missing': 818, 'level3_missing': 2599, 'empty_after_cleaning': 0}
[youtube] level1 value counts:
level1
SUBJECTIVE    3971
NOISE          763
OBJECTIVE       29
Name: 

In [ ]:
task1_test_dataframes = {}
task1_test_load_stats = {}

for key in ["reddit_test", "twitter_test", "youtube_test"]:
    source_name = key.replace("_test", "")
    df, stats = safe_execute(
        load_and_clean_task1_csv,
        TASK1_PATHS[key],
        source_name,
        TASK1_TEST_SOURCE_CONFIG,
        default_return=(None, None),
        log_prefix=f"load_{key}"
    )
    if df is not None:
        task1_test_dataframes[source_name] = df
        task1_test_load_stats[source_name] = stats

print("Loaded Task 1 test sources:", list(task1_test_dataframes.keys()))
for source, df in task1_test_dataframes.items():
    print(f"{source}: {df.shape}")

[reddit] Original: 500 -> Final: 499 rows
[reddit] Duplicates removed: 1
[reddit] Missing report: {'text_missing': 0, 'level1_missing': None, 'level2_missing': None, 'level3_missing': None, 'empty_after_cleaning': 0}
[twitter] Original: 500 -> Final: 409 rows
[twitter] Duplicates removed: 91
[twitter] Missing report: {'text_missing': 0, 'level1_missing': None, 'level2_missing': None, 'level3_missing': None, 'empty_after_cleaning': 0}
[youtube] Original: 500 -> Final: 499 rows
[youtube] Duplicates removed: 1
[youtube] Missing report: {'text_missing': 0, 'level1_missing': None, 'level2_missing': None, 'level3_missing': None, 'empty_after_cleaning': 0}
Loaded Task 1 test sources: ['reddit', 'twitter', 'youtube']
reddit: (499, 13)
twitter: (409, 12)
youtube: (499, 12)


In [ ]:
LEVEL1_CODE_MAP = {0: "NOISE", 1: "OBJECTIVE", 2: "SUBJECTIVE"}
LEVEL2_CODE_MAP = {0: "NEUTRAL", 1: "NEGATIVE", 2: "POSITIVE"}
LEVEL3_CODE_MAP = {0: "NEUTRAL SENTIMENT", 1: "QUESTION", 2: "ADVERTISEMENT", 3: "MISCELLANEOUS"}

TASK1_SOURCE_CONFIG = {
    "reddit": {"text_cols": ["title", "selftext"], "level1": "Level 1", "level2": "Level 2", "level3": "Level 3", "id_col": None},
    "twitter": {"text_cols": ["Tweet"], "level1": "Level 1", "level2": "Level 2", "level3": "Level 3", "id_col": None},
    "youtube": {"text_cols": ["comment"], "level1": "Level 1", "level2": "Level 2", "level3": "Level 3", "id_col": "comment_id"}
}

def code_to_label(value, code_map):
    if pd.isna(value):
        return np.nan
    try:
        code = int(float(value))
    except (ValueError, TypeError):
        return np.nan
    return code_map.get(code, np.nan)

def load_and_clean_task1_csv(filepath, source_name):
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return None, None

    df = None
    for encoding in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(filepath, encoding=encoding)
            break
        except Exception:
            continue

    if df is None:
        print(f"Failed to load {filepath} with any known encoding")
        return None, None

    df.columns = [str(c).strip() for c in df.columns]
    original_len = len(df)

    cfg = TASK1_SOURCE_CONFIG[source_name]
    available_text_cols = [c for c in cfg["text_cols"] if c in df.columns]
    if not available_text_cols:
        print(f"[{source_name}] None of expected text columns {cfg['text_cols']} found in {list(df.columns)}")
        return None, None

    def build_text(row):
        parts = []
        for c in available_text_cols:
            val = row[c]
            if pd.notna(val) and str(val).strip() != "":
                parts.append(str(val))
        return " ".join(parts)

    df["text_raw"] = df.apply(build_text, axis=1)

    level1_col = cfg["level1"] if cfg["level1"] in df.columns else None
    level2_col = cfg["level2"] if cfg["level2"] in df.columns else None
    level3_col = cfg["level3"] if cfg["level3"] in df.columns else None

    df["level1"] = df[level1_col].apply(lambda x: code_to_label(x, LEVEL1_CODE_MAP)) if level1_col else np.nan
    df["level2"] = df[level2_col].apply(lambda x: code_to_label(x, LEVEL2_CODE_MAP)) if level2_col else np.nan
    df["level3"] = df[level3_col].apply(lambda x: code_to_label(x, LEVEL3_CODE_MAP)) if level3_col else np.nan

    if cfg["id_col"] and cfg["id_col"] in df.columns:
        df["record_id"] = df[cfg["id_col"]]
    else:
        df["record_id"] = range(len(df))

    df["text"] = df["text_raw"].apply(clean_text)
    df["is_empty"] = df["text"].apply(is_empty_text)

    missing_report = {
        "text_missing": int(df["text_raw"].isna().sum()),
        "level1_missing": int(df["level1"].isna().sum()),
        "level2_missing": int(df["level2"].isna().sum()),
        "level3_missing": int(df["level3"].isna().sum()),
        "empty_after_cleaning": int(df["is_empty"].sum())
    }

    df = df[~df["is_empty"]].reset_index(drop=True)

    duplicate_count_before = int(df.duplicated(subset=["text"]).sum())
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

    df["source"] = source_name
    df["char_length"] = df["text"].apply(len)
    df["word_count"] = df["text"].apply(lambda x: len(x.split()))

    stats = {
        "source": source_name,
        "original_rows": original_len,
        "final_rows": len(df),
        "duplicates_removed": duplicate_count_before,
        "missing_report": missing_report,
        "text_cols_used": available_text_cols,
        "level_cols_used": [level1_col, level2_col, level3_col]
    }

    print(f"[{source_name}] Original: {original_len} -> Final: {len(df)} rows")
    print(f"[{source_name}] Duplicates removed: {duplicate_count_before}")
    print(f"[{source_name}] Missing report: {missing_report}")
    print(f"[{source_name}] level1 value counts:\n{df['level1'].value_counts(dropna=False)}")

    return df, stats

In [ ]:
def resolve_hierarchy(row):
    if row["level1"] == "OBJECTIVE" and pd.isna(row["level3"]):
        row["level3"] = "NEUTRAL SENTIMENT"
    elif row["level1"] == "NOISE":
        row["level2"] = np.nan
        row["level3"] = np.nan
    return row

for source in task1_dataframes:
    task1_dataframes[source] = task1_dataframes[source].apply(resolve_hierarchy, axis=1)

task1_combined = pd.concat(list(task1_dataframes.values()), ignore_index=True)
print("Combined Task 1 training shape:", task1_combined.shape)
print(task1_combined[["level1", "level2", "level3"]].describe(include="all"))
print("level1 distribution:")
print(task1_combined["level1"].value_counts(dropna=False))
print("level2 distribution:")
print(task1_combined["level2"].value_counts(dropna=False))
print("level3 distribution:")
print(task1_combined["level3"].value_counts(dropna=False))

Combined Task 1 training shape: (13484, 20)
            level1   level2             level3
count        13484     8930               8225
unique           3        3                  4
top     SUBJECTIVE  NEUTRAL  NEUTRAL SENTIMENT
freq          8928     6221               3943
level1 distribution:
level1
SUBJECTIVE    8928
NOISE         2552
OBJECTIVE     2004
Name: count, dtype: int64
level2 distribution:
level2
NEUTRAL     6221
NaN         4554
NEGATIVE    1999
POSITIVE     710
Name: count, dtype: int64
level3 distribution:
level3
NaN                  5259
NEUTRAL SENTIMENT    3943
QUESTION             3459
ADVERTISEMENT         564
MISCELLANEOUS         259
Name: count, dtype: int64


In [ ]:
def compute_basic_eda(df, name):
    report = {}
    report["name"] = name
    report["total_rows"] = len(df)
    report["missing_level1"] = int(df["level1"].isna().sum())
    report["missing_level2"] = int(df["level2"].isna().sum())
    report["missing_level3"] = int(df["level3"].isna().sum())
    report["duplicate_texts"] = int(df["text"].duplicated().sum())
    report["avg_char_length"] = float(df["char_length"].mean())
    report["avg_word_count"] = float(df["word_count"].mean())
    report["median_char_length"] = float(df["char_length"].median())
    report["max_char_length"] = int(df["char_length"].max())
    report["min_char_length"] = int(df["char_length"].min())
    return report

task1_eda_reports = {}
for source, df in task1_dataframes.items():
    report = compute_basic_eda(df, source)
    task1_eda_reports[source] = report
    print(f"--- {source} ---")
    for k, v in report.items():
        print(f"{k}: {v}")
    print()

combined_report = compute_basic_eda(task1_combined, "combined")
task1_eda_reports["combined"] = combined_report
print("--- combined ---")
for k, v in combined_report.items():
    print(f"{k}: {v}")

eda_summary_df = pd.DataFrame(list(task1_eda_reports.values()))
eda_summary_df.to_csv(EDA_DIR / "task1_basic_eda_summary.csv", index=False)
print(eda_summary_df)

--- reddit ---
name: reddit
total_rows: 4962
missing_level1: 0
missing_level2: 1123
missing_level3: 1286
duplicate_texts: 0
avg_char_length: 376.58988311164853
avg_word_count: 65.19407496977026
median_char_length: 117.5
max_char_length: 29260
min_char_length: 1

--- twitter ---
name: twitter
total_rows: 3759
missing_level1: 0
missing_level2: 2639
missing_level3: 1494
duplicate_texts: 0
avg_char_length: 141.3708433093908
avg_word_count: 22.5634477254589
median_char_length: 136.0
max_char_length: 540
min_char_length: 1

--- youtube ---
name: youtube
total_rows: 4763
missing_level1: 0
missing_level2: 792
missing_level3: 2479
duplicate_texts: 0
avg_char_length: 152.52005038841065
avg_word_count: 27.88998530338022
median_char_length: 100.0
max_char_length: 5273
min_char_length: 6

--- combined ---
name: combined
total_rows: 13484
missing_level1: 0
missing_level2: 4554
missing_level3: 5259
duplicate_texts: 0
avg_char_length: 231.86776920795018
avg_word_count: 40.132675763868285
median_char_l

In [ ]:
def plot_label_distribution(df, column, title, save_path):
    counts = df[column].value_counts(dropna=False)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=counts.index.astype(str), y=counts.values)
    plt.title(title)
    plt.xlabel(column)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    return counts

for level in ["level1", "level2", "level3"]:
    counts = plot_label_distribution(
        task1_combined, level, f"Task 1 {level} distribution",
        PLOTS_DIR / f"task1_{level}_distribution.png"
    )
    print(f"--- {level} distribution ---")
    print(counts)
    total = counts.sum()
    proportions = counts / total
    print(f"--- {level} proportions ---")
    print(proportions)
    imbalance_ratio = counts.max() / max(counts.min(), 1)
    print(f"{level} imbalance ratio (max/min): {imbalance_ratio:.2f}")
    print()

--- level1 distribution ---
level1
SUBJECTIVE    8928
NOISE         2552
OBJECTIVE     2004
Name: count, dtype: int64
--- level1 proportions ---
level1
SUBJECTIVE    0.662118
NOISE         0.189261
OBJECTIVE     0.148621
Name: count, dtype: float64
level1 imbalance ratio (max/min): 4.46

--- level2 distribution ---
level2
NEUTRAL     6221
NaN         4554
NEGATIVE    1999
POSITIVE     710
Name: count, dtype: int64
--- level2 proportions ---
level2
NEUTRAL     0.461362
NaN         0.337734
NEGATIVE    0.148250
POSITIVE    0.052655
Name: count, dtype: float64
level2 imbalance ratio (max/min): 8.76

--- level3 distribution ---
level3
NaN                  5259
NEUTRAL SENTIMENT    3943
QUESTION             3459
ADVERTISEMENT         564
MISCELLANEOUS         259
Name: count, dtype: int64
--- level3 proportions ---
level3
NaN                  0.390018
NEUTRAL SENTIMENT    0.292421
QUESTION             0.256526
ADVERTISEMENT        0.041827
MISCELLANEOUS        0.019208
Name: count, dtype: f

In [ ]:
def plot_length_histogram(df, column, title, save_path, bins=50):
    plt.figure(figsize=(10, 6))
    plt.hist(df[column], bins=bins, edgecolor="black")
    plt.title(title)
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

plot_length_histogram(
    task1_combined, "char_length", "Task 1 character length distribution",
    PLOTS_DIR / "task1_char_length_histogram.png"
)
plot_length_histogram(
    task1_combined, "word_count", "Task 1 word count distribution",
    PLOTS_DIR / "task1_word_count_histogram.png"
)

length_percentiles = task1_combined["word_count"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_dict()
print("Word count percentiles:")
for pct, val in length_percentiles.items():
    print(f"p{int(pct*100)}: {val:.2f}")

recommended_max_seq_length = int(min(256, max(32, task1_combined["word_count"].quantile(0.95) * 1.5)))
print(f"Recommended max_seq_length based on p95 word count: {recommended_max_seq_length}")

Word count percentiles:
p50: 20.00
p75: 38.00
p90: 101.70
p95: 150.00
p99: 295.17
Recommended max_seq_length based on p95 word count: 225


In [ ]:
def compute_token_statistics(df, tokenizer_name="roberta-base", sample_size=2000):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    sample_df = df.sample(min(sample_size, len(df)), random_state=GLOBAL_SEED)
    token_lengths = []
    for text in sample_df["text"]:
        tokens = tokenizer.encode(text, truncation=False)
        token_lengths.append(len(tokens))
    token_lengths = np.array(token_lengths)
    stats = {
        "mean_tokens": float(token_lengths.mean()),
        "median_tokens": float(np.median(token_lengths)),
        "max_tokens": int(token_lengths.max()),
        "p90_tokens": float(np.percentile(token_lengths, 90)),
        "p95_tokens": float(np.percentile(token_lengths, 95)),
        "p99_tokens": float(np.percentile(token_lengths, 99))
    }
    del tokenizer
    clear_memory()
    return stats

token_stats = safe_execute(
    compute_token_statistics,
    task1_combined,
    default_return={},
    log_prefix="token_stats"
)
print("Token statistics (roberta-base tokenizer, sample):")
for k, v in token_stats.items():
    print(f"{k}: {v}")

Token indices sequence length is longer than the specified maximum sequence length for this model (702 > 512). Running this sequence through the model will result in indexing errors


Token statistics (roberta-base tokenizer, sample):
mean_tokens: 59.68
median_tokens: 31.0
max_tokens: 1552
p90_tokens: 130.0
p95_tokens: 203.04999999999995
p99_tokens: 385.1099999999999


In [ ]:
def compute_language_statistics(df, sample_size=1000):
    sample_df = df.sample(min(sample_size, len(df)), random_state=GLOBAL_SEED)
    languages = sample_df["text"].apply(detect_language_simple)
    lang_counts = languages.value_counts()
    return lang_counts

language_stats = safe_execute(
    compute_language_statistics,
    task1_combined,
    default_return=pd.Series(dtype=int),
    log_prefix="language_stats"
)
print("Language distribution (sampled):")
print(language_stats)

if len(language_stats) > 0:
    plt.figure(figsize=(10, 6))
    sns.barplot(x=language_stats.index.astype(str), y=language_stats.values)
    plt.title("Language distribution (sampled)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "task1_language_distribution.png")
    plt.close()

Language distribution (sampled):
text
unknown    1000
Name: count, dtype: int64


In [ ]:
label_encoders = {}
for level in ["level1", "level2", "level3"]:
    le = LabelEncoder()
    valid_values = task1_combined[level].dropna().unique().tolist()
    le.fit(valid_values)
    label_encoders[level] = le
    print(level, "classes:", list(le.classes_))

level1 classes: [np.str_('NOISE'), np.str_('OBJECTIVE'), np.str_('SUBJECTIVE')]
level2 classes: [np.str_('NEGATIVE'), np.str_('NEUTRAL'), np.str_('POSITIVE')]
level3 classes: [np.str_('ADVERTISEMENT'), np.str_('MISCELLANEOUS'), np.str_('NEUTRAL SENTIMENT'), np.str_('QUESTION')]


In [ ]:
def make_split(df, level):
    sub = df[df[level].notna()].reset_index(drop=True)
    try:
        train_df, val_df = train_test_split(
            sub, test_size=0.15, random_state=GLOBAL_SEED, stratify=sub[level]
        )
    except ValueError:
        train_df, val_df = train_test_split(sub, test_size=0.15, random_state=GLOBAL_SEED)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

task1_splits = {}
for level in ["level1", "level2", "level3"]:
    train_df, val_df = make_split(task1_combined, level)
    task1_splits[level] = {"train": train_df, "val": val_df}
    print(level, "train:", len(train_df), "val:", len(val_df))

level1 train: 11461 val: 2023
level2 train: 7590 val: 1340
level3 train: 6991 val: 1234


In [ ]:
class Task1Dataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {"text": self.texts[idx], "label": int(self.labels[idx])}

class DynamicPaddingCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        texts = [item["text"] for item in batch]
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        encoded["labels"] = labels
        return encoded

class LengthBucketSampler(torch.utils.data.Sampler):
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = np.argsort(self.lengths)
        buckets = [indices[i:i + self.batch_size * 50] for i in range(0, len(indices), self.batch_size * 50)]
        batches = []
        for bucket in buckets:
            bucket = bucket.copy()
            if self.shuffle:
                np.random.shuffle(bucket)
            for i in range(0, len(bucket), self.batch_size):
                batches.append(bucket[i:i + self.batch_size].tolist())
        if self.shuffle:
            np.random.shuffle(batches)
        for batch in batches:
            for idx in batch:
                yield idx

    def __len__(self):
        return len(self.lengths)

def build_dataloaders(train_df, val_df, level, tokenizer, config):
    y_train = label_encoders[level].transform(train_df[level])
    y_val = label_encoders[level].transform(val_df[level])

    train_ds = Task1Dataset(train_df["text"].tolist(), y_train, tokenizer, config.max_seq_length)
    val_ds = Task1Dataset(val_df["text"].tolist(), y_val, tokenizer, config.max_seq_length)

    collator = DynamicPaddingCollator(tokenizer, config.max_seq_length)
    train_lengths = train_df["word_count"].values

    train_sampler = LengthBucketSampler(train_lengths, config.per_device_batch_size, shuffle=True)

    train_loader = DataLoader(
        train_ds,
        batch_size=config.per_device_batch_size,
        sampler=train_sampler,
        collate_fn=collator,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        persistent_workers=config.persistent_workers if config.num_workers > 0 else False,
        prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=config.per_device_batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        persistent_workers=config.persistent_workers if config.num_workers > 0 else False,
        prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None
    )

    return train_loader, val_loader

In [ ]:
def compute_class_weights(train_df, level):
    y = label_encoders[level].transform(train_df[level])
    counts = np.bincount(y, minlength=len(label_encoders[level].classes_))
    counts = np.where(counts == 0, 1, counts)
    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float)

In [ ]:
from tqdm.auto import tqdm

def evaluate_model(model, val_loader, device, num_classes):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False, mininterval=2.0):
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                outputs = model(**batch)
                logits = outputs.logits
            probs = F.softmax(logits.float(), dim=-1)
            preds = torch.argmax(probs, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, macro_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    _, _, weighted_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    metrics = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }
    return metrics, all_preds, all_labels, all_probs

def train_one_model(model_name, level, config, save_dir):
    save_dir.mkdir(parents=True, exist_ok=True)
    result = {"model_name": model_name, "level": level, "status": "failed"}
    tokenizer = None
    model = None
    try:
        print(f"[{model_name}][{level}] Loading tokenizer...", flush=True)
        num_classes = len(label_encoders[level].classes_)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        print(f"[{model_name}][{level}] Tokenizer loaded. Building dataloaders...", flush=True)
        train_loader, val_loader = build_dataloaders(
            task1_splits[level]["train"], task1_splits[level]["val"], level, tokenizer, config
        )
        print(f"[{model_name}][{level}] Dataloaders built. train_batches={len(train_loader)} val_batches={len(val_loader)}", flush=True)

        print(f"[{model_name}][{level}] Loading model...", flush=True)
        model_config = AutoConfig.from_pretrained(model_name, num_labels=num_classes)
        if hasattr(model_config, "hidden_dropout_prob"):
            model_config.hidden_dropout_prob = config.dropout
        if hasattr(model_config, "attention_probs_dropout_prob"):
            model_config.attention_probs_dropout_prob = config.dropout

        model = AutoModelForSequenceClassification.from_pretrained(model_name, config=model_config)
        if config.use_gradient_checkpointing:
            try:
                model.gradient_checkpointing_enable()
            except Exception:
                pass
        model.to(DEVICE)
        print(f"[{model_name}][{level}] Model loaded on {DEVICE}. Starting training...", flush=True)

        class_weights = compute_class_weights(task1_splits[level]["train"], level).to(DEVICE)
        loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config.label_smoothing)

        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": config.weight_decay},
            {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}
        ]
        optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=config.learning_rate)

        total_steps = (len(train_loader) // config.gradient_accumulation_steps) * config.num_epochs
        warmup_steps = int(total_steps * config.warmup_ratio)
        scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

        scaler = GradScaler(enabled=HW.use_amp and HW.amp_dtype == torch.float16)

        best_macro_f1 = -1
        patience_counter = 0
        history = []
        start_time = time.time()

        for epoch in range(config.num_epochs):
            model.train()
            epoch_loss = 0.0
            optimizer.zero_grad()
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}", mininterval=2.0)
            for step, batch in enumerate(progress_bar):
                labels = batch.pop("labels").to(DEVICE)
                batch = {k: v.to(DEVICE) for k, v in batch.items()}

                def train_step(batch_size_unused):
                    with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                        outputs = model(**batch)
                        loss = loss_fn(outputs.logits, labels)
                        loss_scaled = loss / config.gradient_accumulation_steps
                    if scaler.is_enabled():
                        scaler.scale(loss_scaled).backward()
                    else:
                        loss_scaled.backward()
                    return loss.item()

                try:
                    loss_val = train_step(config.per_device_batch_size)
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print(f"OOM during training step, clearing memory and skipping batch", flush=True)
                        optimizer.zero_grad()
                        clear_memory()
                        continue
                    else:
                        raise

                epoch_loss += loss_val
                progress_bar.set_postfix({"loss": f"{loss_val:.4f}"})

                if (step + 1) % config.gradient_accumulation_steps == 0:
                    if scaler.is_enabled():
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                        optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            avg_train_loss = epoch_loss / max(1, len(train_loader))
            val_metrics, _, _, _ = evaluate_model(model, val_loader, DEVICE, num_classes)
            history.append({"epoch": epoch + 1, "train_loss": avg_train_loss, **val_metrics})
            print(f"[{model_name}][{level}] Epoch {epoch+1}/{config.num_epochs} train_loss={avg_train_loss:.4f} macro_f1={val_metrics['macro_f1']:.4f}", flush=True)

            if val_metrics["macro_f1"] > best_macro_f1:
                best_macro_f1 = val_metrics["macro_f1"]
                patience_counter = 0
                model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)
                with open(save_dir / "best_metrics.json", "w") as f:
                    json.dump(val_metrics, f, indent=2)
            else:
                patience_counter += 1
                if patience_counter >= config.early_stopping_patience:
                    print(f"Early stopping at epoch {epoch+1}", flush=True)
                    break

            clear_memory()

        training_time = time.time() - start_time
        gpu_stats = get_gpu_memory_stats()

        with open(save_dir / "training_history.json", "w") as f:
            json.dump(history, f, indent=2)

        result.update({
            "status": "success",
            "best_macro_f1": best_macro_f1,
            "training_time_sec": training_time,
            "gpu_stats": gpu_stats,
            "save_dir": str(save_dir),
            "num_classes": num_classes
        })

    except Exception as e:
        print(f"Training failed for {model_name} on {level}: {e}", flush=True)
        traceback.print_exc()
        result["error"] = str(e)
    finally:
        delete_model_safely(model, tokenizer)

    return result

In [ ]:
TASK1_MODEL_ZOO = [
    "microsoft/deberta-v3-base",
    "roberta-base",
    "distilroberta-base"
]

print("Task 1 model zoo (trimmed for faster iteration):")
for m in TASK1_MODEL_ZOO:
    print(m)

Task 1 model zoo (trimmed for faster iteration):
microsoft/deberta-v3-base
roberta-base
distilroberta-base


In [ ]:
HYPERPARAM_SEARCH_SPACE = [
    {"learning_rate": 2e-5, "per_device_batch_size": 16, "max_seq_length": 128, "dropout": 0.1, "weight_decay": 0.01, "warmup_ratio": 0.06, "num_epochs": 3}
]

def build_config_from_params(params, gradient_accumulation_steps=2):
    cfg = TrainingConfig(
        learning_rate=params["learning_rate"],
        per_device_batch_size=params["per_device_batch_size"],
        gradient_accumulation_steps=gradient_accumulation_steps,
        max_seq_length=params["max_seq_length"],
        dropout=params["dropout"],
        weight_decay=params["weight_decay"],
        warmup_ratio=params["warmup_ratio"],
        num_epochs=params["num_epochs"]
    )
    return cfg

print(f"{len(HYPERPARAM_SEARCH_SPACE)} hyperparameter configuration(s) per model per level")

1 hyperparameter configuration(s) per model per level


In [ ]:
def sanitize_name(name):
    return name.replace("/", "_").replace("-", "_")

all_experiments = []

for level in ["level1", "level2", "level3"]:
    for model_name in TASK1_MODEL_ZOO:
        for param_idx, params in enumerate(HYPERPARAM_SEARCH_SPACE):
            log_gpu_usage(f"before_{model_name}_{level}_cfg{param_idx}")
            config = build_config_from_params(params)
            exp_dir = CHECKPOINT_DIR / "task1" / level / f"{sanitize_name(model_name)}_cfg{param_idx}"

            result = safe_execute(
                train_one_model,
                model_name,
                level,
                config,
                exp_dir,
                default_return={"model_name": model_name, "level": level, "status": "failed", "error": "unhandled exception"},
                log_prefix=f"{model_name}_{level}_cfg{param_idx}"
            )
            result["config_index"] = param_idx
            result["hyperparams"] = params
            all_experiments.append(result)

            clear_memory()
            log_gpu_usage(f"after_{model_name}_{level}_cfg{param_idx}")

with open(METRICS_DIR / "task1_all_experiments.json", "w") as f:
    json.dump(all_experiments, f, indent=2, default=str)

print(f"Total experiments run: {len(all_experiments)}")
print(f"Successful: {sum(1 for e in all_experiments if e['status'] == 'success')}")
print(f"Failed: {sum(1 for e in all_experiments if e['status'] == 'failed')}")

[before_microsoft/deberta-v3-base_level1_cfg0] Allocated: 2.851GB | Reserved: 3.053GB | Free: 11.789GB | Total: 14.842GB
[microsoft/deberta-v3-base][level1] Loading tokenizer...
[microsoft/deberta-v3-base][level1] Tokenizer loaded. Building dataloaders...
[microsoft/deberta-v3-base][level1] Dataloaders built. train_batches=717 val_batches=127
[microsoft/deberta-v3-base][level1] Loading model...


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[microsoft/deberta-v3-base][level1] Model loaded on cuda. Starting training...


Epoch 1/3:   0%|          | 0/717 [00:00<?, ?it/s]

Validating:   0%|          | 0/127 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/78 [00:00<?, ?it/s]

[distilroberta-base][level3] Epoch 2/3 train_loss=0.9468 macro_f1=0.6572


Epoch 3/3:   0%|          | 0/437 [00:00<?, ?it/s]

Validating:   0%|          | 0/78 [00:00<?, ?it/s]

[distilroberta-base][level3] Epoch 3/3 train_loss=0.8484 macro_f1=0.6885
[after_distilroberta-base_level3_cfg0] Allocated: 2.851GB | Reserved: 3.053GB | Free: 11.789GB | Total: 14.842GB
Total experiments run: 9
Successful: 9
Failed: 0


In [ ]:
def build_experiment_comparison_table(experiments):
    rows = []
    for exp in experiments:
        if exp["status"] != "success":
            continue
        rows.append({
            "model_name": exp["model_name"],
            "level": exp["level"],
            "config_index": exp["config_index"],
            "macro_f1": exp["best_macro_f1"],
            "training_time_sec": exp["training_time_sec"],
            "learning_rate": exp["hyperparams"]["learning_rate"],
            "batch_size": exp["hyperparams"]["per_device_batch_size"],
            "max_seq_length": exp["hyperparams"]["max_seq_length"],
            "epochs": exp["hyperparams"]["num_epochs"],
            "save_dir": exp["save_dir"]
        })
    return pd.DataFrame(rows).sort_values(["level", "macro_f1"], ascending=[True, False])

task1_comparison_df = build_experiment_comparison_table(all_experiments)
task1_comparison_df.to_csv(METRICS_DIR / "task1_experiment_comparison.csv", index=False)
print(task1_comparison_df.to_string())

task1_best_per_level = {}
for level in ["level1", "level2", "level3"]:
    level_df = task1_comparison_df[task1_comparison_df["level"] == level]
    if len(level_df) == 0:
        print(f"No successful experiments for {level}")
        continue
    best_row = level_df.iloc[0]
    task1_best_per_level[level] = best_row.to_dict()
    print(f"Best model for {level}: {best_row['model_name']} cfg{best_row['config_index']} macro_f1={best_row['macro_f1']:.4f}")

with open(METRICS_DIR / "task1_best_per_level.json", "w") as f:
    json.dump(task1_best_per_level, f, indent=2, default=str)

                  model_name   level  config_index  macro_f1  training_time_sec  learning_rate  batch_size  max_seq_length  epochs                                                                                        save_dir
1               roberta-base  level1             0  0.768397         453.846711        0.00002          16             128       3               C:\Users\VGST\Desktop\CryptoQA\outputs\checkpoints\task1\level1\roberta_base_cfg0
0  microsoft/deberta-v3-base  level1             0  0.754648        1086.020858        0.00002          16             128       3  C:\Users\VGST\Desktop\CryptoQA\outputs\checkpoints\task1\level1\microsoft_deberta_v3_base_cfg0
2         distilroberta-base  level1             0  0.754445         227.602839        0.00002          16             128       3         C:\Users\VGST\Desktop\CryptoQA\outputs\checkpoints\task1\level1\distilroberta_base_cfg0
4               roberta-base  level2             0  0.699934         242.912343        0.000

In [ ]:
def full_evaluation_report(model_name, level, save_dir, config):
    tokenizer = None
    model = None
    try:
        num_classes = len(label_encoders[level].classes_)
        tokenizer = AutoTokenizer.from_pretrained(save_dir)
        model = AutoModelForSequenceClassification.from_pretrained(save_dir)
        model.to(DEVICE)

        _, val_loader = build_dataloaders(
            task1_splits[level]["train"], task1_splits[level]["val"], level, tokenizer, config
        )

        metrics, preds, labels, probs = evaluate_model(model, val_loader, DEVICE, num_classes)

        cm = confusion_matrix(labels, preds, labels=list(range(num_classes)))
        class_names = label_encoders[level].classes_

        report = classification_report(
            labels, preds, labels=list(range(num_classes)), target_names=class_names,
            zero_division=0, output_dict=True
        )

        probs_arr = np.array(probs)
        roc_auc = None
        try:
            if num_classes == 2:
                roc_auc = roc_auc_score(labels, probs_arr[:, 1])
            else:
                roc_auc = roc_auc_score(labels, probs_arr, multi_class="ovr", average="macro")
        except Exception as e:
            print(f"ROC AUC could not be computed for {model_name}/{level}: {e}")

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
        plt.title(f"Confusion Matrix - {model_name} - {level}")
        plt.ylabel("True")
        plt.xlabel("Predicted")
        plt.tight_layout()
        cm_path = PLOTS_DIR / f"cm_{sanitize_name(model_name)}_{level}.png"
        plt.savefig(cm_path)
        plt.close()

        with open(METRICS_DIR / f"classification_report_{sanitize_name(model_name)}_{level}.json", "w") as f:
            json.dump(report, f, indent=2, default=str)

        result = {
            "model_name": model_name,
            "level": level,
            "metrics": metrics,
            "roc_auc": roc_auc,
            "confusion_matrix_path": str(cm_path),
            "classification_report": report,
            "val_preds": preds,
            "val_labels": labels,
            "val_probs": probs
        }
        return result
    except Exception as e:
        print(f"Full evaluation failed for {model_name}/{level}: {e}")
        traceback.print_exc()
        return None
    finally:
        delete_model_safely(model, tokenizer)

task1_full_eval_results = {}
for level, best in task1_best_per_level.items():
    config = build_config_from_params(HYPERPARAM_SEARCH_SPACE[best["config_index"]])
    result = safe_execute(
        full_evaluation_report,
        best["model_name"],
        level,
        Path(best["save_dir"]),
        config,
        default_return=None,
        log_prefix=f"full_eval_{level}"
    )
    if result is not None:
        task1_full_eval_results[level] = result
        print(f"--- {level} full evaluation ---")
        print("Metrics:", result["metrics"])
        print("ROC AUC:", result["roc_auc"])
    clear_memory()

Validating:   0%|          | 0/127 [00:00<?, ?it/s]

--- level1 full evaluation ---
Metrics: {'accuracy': 0.819080573405833, 'precision': 0.7438558718370164, 'recall': 0.805068101279813, 'macro_f1': 0.7683966122570046, 'weighted_f1': 0.825290084706065}
ROC AUC: 0.9309052823033618


Validating:   0%|          | 0/84 [00:00<?, ?it/s]

--- level2 full evaluation ---
Metrics: {'accuracy': 0.8014925373134328, 'precision': 0.6791551131016349, 'recall': 0.7511626315139251, 'macro_f1': 0.6999339589949836, 'weighted_f1': 0.8155114445584627}
ROC AUC: 0.8947936718327515


Validating:   0%|          | 0/78 [00:00<?, ?it/s]

--- level3 full evaluation ---
Metrics: {'accuracy': 0.8354943273905997, 'precision': 0.6738817033442062, 'recall': 0.7686508381704256, 'macro_f1': 0.7067889270409478, 'weighted_f1': 0.8440950132742909}
ROC AUC: 0.9382266614440636


In [ ]:
def get_top_n_models_per_level(comparison_df, level, n=3):
    level_df = comparison_df[comparison_df["level"] == level].sort_values("macro_f1", ascending=False)
    return level_df.head(n).to_dict("records")

task1_ensemble_candidates = {}
for level in ["level1", "level2", "level3"]:
    candidates = get_top_n_models_per_level(task1_comparison_df, level, n=3)
    task1_ensemble_candidates[level] = candidates
    print(f"--- Top candidates for {level} ---")
    for c in candidates:
        print(c["model_name"], "cfg", c["config_index"], "macro_f1", round(c["macro_f1"], 4))

--- Top candidates for level1 ---
roberta-base cfg 0 macro_f1 0.7684
microsoft/deberta-v3-base cfg 0 macro_f1 0.7546
distilroberta-base cfg 0 macro_f1 0.7544
--- Top candidates for level2 ---
roberta-base cfg 0 macro_f1 0.6999
microsoft/deberta-v3-base cfg 0 macro_f1 0.6729
distilroberta-base cfg 0 macro_f1 0.6714
--- Top candidates for level3 ---
roberta-base cfg 0 macro_f1 0.7068
distilroberta-base cfg 0 macro_f1 0.6885
microsoft/deberta-v3-base cfg 0 macro_f1 0.6816


In [ ]:
def get_val_probs_for_candidate(candidate, level):
    save_dir = Path(candidate["save_dir"])
    config = build_config_from_params(HYPERPARAM_SEARCH_SPACE[candidate["config_index"]])
    tokenizer = None
    model = None
    try:
        num_classes = len(label_encoders[level].classes_)
        tokenizer = AutoTokenizer.from_pretrained(save_dir)
        model = AutoModelForSequenceClassification.from_pretrained(save_dir)
        model.to(DEVICE)
        _, val_loader = build_dataloaders(
            task1_splits[level]["train"], task1_splits[level]["val"], level, tokenizer, config
        )
        metrics, preds, labels, probs = evaluate_model(model, val_loader, DEVICE, num_classes)
        return np.array(probs), np.array(labels), metrics["macro_f1"]
    except Exception as e:
        print(f"Failed getting val probs for {candidate['model_name']}/{level}: {e}")
        traceback.print_exc()
        return None, None, None
    finally:
        delete_model_safely(model, tokenizer)

task1_val_probs = {}
task1_val_labels_by_level = {}
for level, candidates in task1_ensemble_candidates.items():
    task1_val_probs[level] = []
    for cand in candidates:
        probs, labels, f1 = safe_execute(
            get_val_probs_for_candidate, cand, level,
            default_return=(None, None, None),
            log_prefix=f"val_probs_{level}_{cand['model_name']}"
        )
        if probs is not None:
            task1_val_probs[level].append({"model_name": cand["model_name"], "probs": probs, "macro_f1": f1})
            task1_val_labels_by_level[level] = labels
        clear_memory()
    print(f"{level}: collected val probs for {len(task1_val_probs[level])} models")

Validating:   0%|          | 0/127 [00:00<?, ?it/s]

Validating:   0%|          | 0/127 [00:00<?, ?it/s]

Validating:   0%|          | 0/127 [00:00<?, ?it/s]

level1: collected val probs for 3 models


Validating:   0%|          | 0/84 [00:00<?, ?it/s]

...

level2: collected val probs for 3 models


Validating:   0%|          | 0/78 [00:00<?, ?it/s]

Validating:   0%|          | 0/78 [00:00<?, ?it/s]

Validating:   0%|          | 0/78 [00:00<?, ?it/s]

level3: collected val probs for 3 models


In [ ]:
def hard_voting(prob_list):
    preds_list = [np.argmax(p["probs"], axis=1) for p in prob_list]
    preds_stack = np.stack(preds_list, axis=1)
    final_preds = []
    for row in preds_stack:
        counts = np.bincount(row)
        final_preds.append(np.argmax(counts))
    return np.array(final_preds)

def soft_voting(prob_list):
    avg_probs = np.mean([p["probs"] for p in prob_list], axis=0)
    return np.argmax(avg_probs, axis=1), avg_probs

def weighted_average(prob_list):
    weights = np.array([p["macro_f1"] for p in prob_list])
    weights = weights / weights.sum()
    weighted_probs = np.zeros_like(prob_list[0]["probs"])
    for w, p in zip(weights, prob_list):
        weighted_probs += w * p["probs"]
    return np.argmax(weighted_probs, axis=1), weighted_probs

def rank_averaging(prob_list):
    rank_sum = np.zeros_like(prob_list[0]["probs"])
    for p in prob_list:
        ranks = np.argsort(np.argsort(p["probs"], axis=1), axis=1)
        rank_sum += ranks
    return np.argmax(rank_sum, axis=1), rank_sum

def evaluate_ensemble_preds(preds, labels):
    acc = accuracy_score(labels, preds)
    precision, recall, macro_f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    _, _, weighted_f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": precision, "recall": recall, "macro_f1": macro_f1, "weighted_f1": weighted_f1}

task1_ensemble_results = {}
for level, prob_list in task1_val_probs.items():
    if len(prob_list) < 2:
        print(f"Skipping ensembling for {level}, fewer than 2 successful models")
        continue
    labels = task1_val_labels_by_level[level]

    hard_preds = hard_voting(prob_list)
    soft_preds, soft_probs = soft_voting(prob_list)
    weighted_preds, weighted_probs = weighted_average(prob_list)
    rank_preds, _ = rank_averaging(prob_list)

    results = {
        "hard_voting": evaluate_ensemble_preds(hard_preds, labels),
        "soft_voting": evaluate_ensemble_preds(soft_preds, labels),
        "weighted_average": evaluate_ensemble_preds(weighted_preds, labels),
        "rank_averaging": evaluate_ensemble_preds(rank_preds, labels)
    }
    task1_ensemble_results[level] = results
    print(f"--- Ensemble results for {level} ---")
    for strategy, metrics in results.items():
        print(strategy, metrics)

--- Ensemble results for level1 ---
hard_voting {'accuracy': 0.8215521502718734, 'precision': 0.744682661037188, 'recall': 0.8011373148638986, 'macro_f1': 0.7658599152485964, 'weighted_f1': 0.827024684180189}
soft_voting {'accuracy': 0.8200692041522492, 'precision': 0.7439067936389997, 'recall': 0.8042086820662869, 'macro_f1': 0.7662397029127664, 'weighted_f1': 0.8258285012248175}
weighted_average {'accuracy': 0.8200692041522492, 'precision': 0.7439067936389997, 'recall': 0.8042086820662869, 'macro_f1': 0.7662397029127664, 'weighted_f1': 0.8258285012248175}
rank_averaging {'accuracy': 0.8210578348986654, 'precision': 0.7445368009249003, 'recall': 0.8002669928447514, 'macro_f1': 0.7653163620120292, 'weighted_f1': 0.8264266989843776}
--- Ensemble results for level2 ---
hard_voting {'accuracy': 0.7858208955223881, 'precision': 0.6635721409620189, 'recall': 0.7426670072422393, 'macro_f1': 0.6847396989110585, 'weighted_f1': 0.8030683017234777}
soft_voting {'accuracy': 0.7895522388059701, 'p

In [ ]:
def stacking_ensemble(prob_list, labels):
    stacked_features = np.concatenate([p["probs"] for p in prob_list], axis=1)
    train_idx, val_idx = train_test_split(
        np.arange(len(labels)), test_size=0.3, random_state=GLOBAL_SEED, stratify=labels
    )
    meta_model = LogisticRegression(max_iter=1000, multi_class="auto")
    meta_model.fit(stacked_features[train_idx], labels[train_idx])
    meta_preds = meta_model.predict(stacked_features[val_idx])
    meta_metrics = evaluate_ensemble_preds(meta_preds, labels[val_idx])
    return meta_model, meta_metrics

task1_stacking_results = {}
task1_stacking_models = {}
for level, prob_list in task1_val_probs.items():
    if len(prob_list) < 2:
        continue
    labels = task1_val_labels_by_level[level]
    meta_model, meta_metrics = safe_execute(
        stacking_ensemble, prob_list, labels,
        default_return=(None, None),
        log_prefix=f"stacking_{level}"
    )
    if meta_model is not None:
        task1_stacking_models[level] = meta_model
        task1_stacking_results[level] = meta_metrics
        print(f"--- Stacking results for {level} ---")
        print(meta_metrics)

--- Stacking results for level1 ---
{'accuracy': 0.8401976935749588, 'precision': 0.7837551687341603, 'recall': 0.8011488451462494, 'macro_f1': 0.792061683099304, 'weighted_f1': 0.8413986969725128}
--- Stacking results for level2 ---
{'accuracy': 0.8731343283582089, 'precision': 0.7748696111441209, 'recall': 0.7078373015873015, 'macro_f1': 0.7329213247757441, 'weighted_f1': 0.8653574453517545}
--- Stacking results for level3 ---
{'accuracy': 0.8867924528301887, 'precision': 0.795240597440033, 'recall': 0.7174697493517719, 'macro_f1': 0.7462401458452025, 'weighted_f1': 0.8833476222437837}


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

def select_best_ensemble_strategy(level):
    candidates = {}
    if level in task1_ensemble_results:
        for strategy, metrics in task1_ensemble_results[level].items():
            candidates[strategy] = metrics["macro_f1"]
    if level in task1_stacking_results and task1_stacking_results[level] is not None:
        candidates["stacking"] = task1_stacking_results[level]["macro_f1"]
    if not candidates:
        return None, None
    best_strategy = max(candidates, key=candidates.get)
    return best_strategy, candidates[best_strategy]

task1_best_ensemble_strategy = {}
for level in task1_val_probs:
    strategy, score = select_best_ensemble_strategy(level)
    task1_best_ensemble_strategy[level] = {"strategy": strategy, "macro_f1": score}
    print(f"Best ensemble strategy for {level}: {strategy} (macro_f1={score})")

def build_calibrated_ensemble(prob_list, labels):
    stacked_features = np.concatenate([p["probs"] for p in prob_list], axis=1)
    train_idx, val_idx = train_test_split(
        np.arange(len(labels)), test_size=0.3, random_state=GLOBAL_SEED, stratify=labels
    )
    base_lr = LogisticRegression(max_iter=1000)
    calibrated = CalibratedClassifierCV(base_lr, method="sigmoid", cv=3)
    calibrated.fit(stacked_features[train_idx], labels[train_idx])
    preds = calibrated.predict(stacked_features[val_idx])
    metrics = evaluate_ensemble_preds(preds, labels[val_idx])
    return calibrated, metrics

task1_calibrated_ensembles = {}
task1_calibrated_metrics = {}
for level, prob_list in task1_val_probs.items():
    if len(prob_list) < 2:
        continue
    labels = task1_val_labels_by_level[level]
    calibrated_model, calibrated_metrics = safe_execute(
        build_calibrated_ensemble, prob_list, labels,
        default_return=(None, None),
        log_prefix=f"calibrated_{level}"
    )
    if calibrated_model is not None:
        task1_calibrated_ensembles[level] = calibrated_model
        task1_calibrated_metrics[level] = calibrated_metrics
        print(f"--- Calibrated ensemble for {level} ---")
        print(calibrated_metrics)

with open(METRICS_DIR / "task1_ensemble_summary.json", "w") as f:
    json.dump({
        "voting_results": task1_ensemble_results,
        "stacking_results": task1_stacking_results,
        "calibrated_results": task1_calibrated_metrics,
        "best_strategy_per_level": task1_best_ensemble_strategy
    }, f, indent=2, default=str)

Best ensemble strategy for level1: stacking (macro_f1=0.792061683099304)
Best ensemble strategy for level2: stacking (macro_f1=0.7329213247757441)
Best ensemble strategy for level3: stacking (macro_f1=0.7462401458452025)
--- Calibrated ensemble for level1 ---
{'accuracy': 0.8385502471169687, 'precision': 0.7766989621763266, 'recall': 0.803999327036316, 'macro_f1': 0.7888880693412917, 'weighted_f1': 0.8398498873789864}
--- Calibrated ensemble for level2 ---
{'accuracy': 0.8706467661691543, 'precision': 0.7660181189592955, 'recall': 0.7066468253968253, 'macro_f1': 0.7296346211440551, 'weighted_f1': 0.8634142852672843}
--- Calibrated ensemble for level3 ---
{'accuracy': 0.8787061994609164, 'precision': 0.7642237769558239, 'recall': 0.7132562662057044, 'macro_f1': 0.7334098226537629, 'weighted_f1': 0.8759346026709849}


In [ ]:
task1_test_source_map = {
    "twitter": "tweet",
    "reddit": "reddit",
    "youtube": "youtube"
}

for source, df in task1_test_dataframes.items():
    print(source, df.shape)
    print(df[["record_id", "text"]].head(3))

reddit (499, 13)
   record_id                                               text
0          0  Where to buy crypto using Indian credit card?B...
1          1                                    Money vanished?
2          2                                             Bitrue
twitter (409, 12)
   record_id                                               text
0          0  rt @upark_network: upark presents a mobile-bas...
1          1  rt @themanoban_walk: ready to start the new we...
2          2  rt @eithanjames1214: something coming up on @b...
youtube (499, 12)
      record_id                                               text
0   video_35661  This video was a waste of time. The things he ...
1  video_109024  Knowledge is power and a low IQ without knowle...
2  video_219261  This happens already on 2019 to 2021 it was 3k...


In [ ]:
def run_inference_batches(model, tokenizer, texts, max_length, device, batch_size=32):
    all_probs = []

    def infer_batch(bs):
        results = []
        for i in range(0, len(texts), bs):
            chunk = texts[i:i + bs]
            encoded = tokenizer(
                chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt"
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            with torch.no_grad():
                with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                    outputs = model(**encoded)
                    logits = outputs.logits
            probs = F.softmax(logits.float(), dim=-1).cpu().numpy()
            results.append(probs)
        return np.concatenate(results, axis=0)

    probs, _ = OOM_RUNNER.run(infer_batch, batch_size)
    return probs

def load_checkpoint_for_inference(save_dir):
    tokenizer = AutoTokenizer.from_pretrained(save_dir)
    model = AutoModelForSequenceClassification.from_pretrained(save_dir)
    model.to(DEVICE)
    model.eval()
    return model, tokenizer

In [ ]:
def predict_level_for_source(level, texts, strategy_info):
    if strategy_info is None:
        return None
    save_dir = Path(strategy_info["save_dir"])
    max_length = HYPERPARAM_SEARCH_SPACE[strategy_info["config_index"]]["max_seq_length"]
    model = None
    tokenizer = None
    try:
        model, tokenizer = load_checkpoint_for_inference(save_dir)
        probs = run_inference_batches(model, tokenizer, texts, max_length, DEVICE)
        preds = np.argmax(probs, axis=1)
        labels_text = label_encoders[level].inverse_transform(preds)
        return labels_text, probs
    except Exception as e:
        print(f"Prediction failed for {level} using best single model: {e}")
        traceback.print_exc()
        return None, None
    finally:
        delete_model_safely(model, tokenizer)

def generate_submission_1():
    out_dir = SUBMISSION_DIRS_TASK1[1]
    filename_map = {"twitter": "crypto_test_tweet.csv", "reddit": "crypto_test_reddit.csv", "youtube": "crypto_test_youtube.csv"}

    for source, df in task1_test_dataframes.items():
        texts = df["text"].tolist()
        output_df = pd.DataFrame({"id": df["record_id"]})

        for level in ["level1", "level2", "level3"]:
            best = task1_best_per_level.get(level)
            labels_text, probs = safe_execute(
                predict_level_for_source, level, texts, best,
                default_return=(None, None),
                log_prefix=f"sub1_{source}_{level}"
            )
            if labels_text is not None:
                output_df[level] = labels_text
            clear_memory()

        out_path = out_dir / filename_map[source]
        output_df.to_csv(out_path, index=False)
        print(f"Saved Submission 1 for {source} to {out_path}")

generate_submission_1()

Saved Submission 1 for reddit to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_1\crypto_test_reddit.csv
Saved Submission 1 for twitter to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_1\crypto_test_tweet.csv
Saved Submission 1 for youtube to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_1\crypto_test_youtube.csv


In [ ]:
def predict_ensemble_for_source(level, texts):
    candidates = task1_ensemble_candidates.get(level, [])
    if len(candidates) < 2:
        return None

    strategy = task1_best_ensemble_strategy.get(level, {}).get("strategy")
    if strategy is None:
        return None

    per_model_probs = []
    for cand in candidates:
        save_dir = Path(cand["save_dir"])
        max_length = HYPERPARAM_SEARCH_SPACE[cand["config_index"]]["max_seq_length"]
        model = None
        tokenizer = None
        try:
            model, tokenizer = load_checkpoint_for_inference(save_dir)
            probs = run_inference_batches(model, tokenizer, texts, max_length, DEVICE)
            per_model_probs.append({"model_name": cand["model_name"], "probs": probs, "macro_f1": cand["macro_f1"]})
        except Exception as e:
            print(f"Ensemble sub-model prediction failed for {cand['model_name']}/{level}: {e}")
            traceback.print_exc()
        finally:
            delete_model_safely(model, tokenizer)
        clear_memory()

    if len(per_model_probs) < 2:
        return None

    if strategy == "hard_voting":
        preds = hard_voting(per_model_probs)
    elif strategy == "soft_voting":
        preds, _ = soft_voting(per_model_probs)
    elif strategy == "weighted_average":
        preds, _ = weighted_average(per_model_probs)
    elif strategy == "rank_averaging":
        preds, _ = rank_averaging(per_model_probs)
    elif strategy == "stacking":
        meta_model = task1_stacking_models.get(level)
        if meta_model is None:
            return None
        stacked_features = np.concatenate([p["probs"] for p in per_model_probs], axis=1)
        preds = meta_model.predict(stacked_features)
    else:
        return None

    labels_text = label_encoders[level].inverse_transform(preds)
    return labels_text

def generate_submission_2():
    out_dir = SUBMISSION_DIRS_TASK1[2]
    filename_map = {"twitter": "crypto_test_tweet.csv", "reddit": "crypto_test_reddit.csv", "youtube": "crypto_test_youtube.csv"}

    for source, df in task1_test_dataframes.items():
        texts = df["text"].tolist()
        output_df = pd.DataFrame({"id": df["record_id"]})

        for level in ["level1", "level2", "level3"]:
            labels_text = safe_execute(
                predict_ensemble_for_source, level, texts,
                default_return=None,
                log_prefix=f"sub2_{source}_{level}"
            )
            if labels_text is not None:
                output_df[level] = labels_text
            else:
                best = task1_best_per_level.get(level)
                fallback_labels, _ = safe_execute(
                    predict_level_for_source, level, texts, best,
                    default_return=(None, None),
                    log_prefix=f"sub2_fallback_{source}_{level}"
                )
                if fallback_labels is not None:
                    output_df[level] = fallback_labels
            clear_memory()

        out_path = out_dir / filename_map[source]
        output_df.to_csv(out_path, index=False)
        print(f"Saved Submission 2 for {source} to {out_path}")

generate_submission_2()

Saved Submission 2 for reddit to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_2\crypto_test_reddit.csv
Saved Submission 2 for twitter to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_2\crypto_test_tweet.csv
Saved Submission 2 for youtube to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_2\crypto_test_youtube.csv


In [ ]:
def predict_calibrated_for_source(level, texts):
    candidates = task1_ensemble_candidates.get(level, [])
    calibrated_model = task1_calibrated_ensembles.get(level)
    if len(candidates) < 2 or calibrated_model is None:
        return None

    per_model_probs = []
    for cand in candidates:
        save_dir = Path(cand["save_dir"])
        max_length = HYPERPARAM_SEARCH_SPACE[cand["config_index"]]["max_seq_length"]
        model = None
        tokenizer = None
        try:
            model, tokenizer = load_checkpoint_for_inference(save_dir)
            probs = run_inference_batches(model, tokenizer, texts, max_length, DEVICE)
            per_model_probs.append(probs)
        except Exception as e:
            print(f"Calibrated ensemble sub-model prediction failed for {cand['model_name']}/{level}: {e}")
            traceback.print_exc()
        finally:
            delete_model_safely(model, tokenizer)
        clear_memory()

    if len(per_model_probs) < 2:
        return None

    stacked_features = np.concatenate(per_model_probs, axis=1)
    preds = calibrated_model.predict(stacked_features)
    labels_text = label_encoders[level].inverse_transform(preds)
    return labels_text

def generate_submission_3():
    out_dir = SUBMISSION_DIRS_TASK1[3]
    filename_map = {"twitter": "crypto_test_tweet.csv", "reddit": "crypto_test_reddit.csv", "youtube": "crypto_test_youtube.csv"}

    for source, df in task1_test_dataframes.items():
        texts = df["text"].tolist()
        output_df = pd.DataFrame({"id": df["record_id"]})

        for level in ["level1", "level2", "level3"]:
            labels_text = safe_execute(
                predict_calibrated_for_source, level, texts,
                default_return=None,
                log_prefix=f"sub3_{source}_{level}"
            )
            if labels_text is not None:
                output_df[level] = labels_text
            else:
                best = task1_best_per_level.get(level)
                fallback_labels, _ = safe_execute(
                    predict_level_for_source, level, texts, best,
                    default_return=(None, None),
                    log_prefix=f"sub3_fallback_{source}_{level}"
                )
                if fallback_labels is not None:
                    output_df[level] = fallback_labels
            clear_memory()

        out_path = out_dir / filename_map[source]
        output_df.to_csv(out_path, index=False)
        print(f"Saved Submission 3 for {source} to {out_path}")

generate_submission_3()

Saved Submission 3 for reddit to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_3\crypto_test_reddit.csv
Saved Submission 3 for twitter to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_3\crypto_test_tweet.csv
Saved Submission 3 for youtube to C:\Users\VGST\Desktop\CryptoQA\outputs\Task1\Submission_3\crypto_test_youtube.csv


Task2

In [ ]:
import csv

def load_jsonl(filepath):
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return None
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    if not records:
        return None
    return pd.DataFrame(records)

def load_csv_flexible(filepath):
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return None

    for encoding in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(filepath, encoding=encoding)
            print(f"Loaded {filepath.name} with encoding={encoding} (C engine), shape={df.shape}")
            return df
        except Exception as e:
            print(f"C engine failed with encoding={encoding}: {type(e).__name__}: {e}")

    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(
                filepath,
                encoding=encoding,
                engine="python",
                on_bad_lines="skip"
            )
            print(f"Loaded {filepath.name} with encoding={encoding} (python engine, bad lines skipped), shape={df.shape}")
            return df
        except Exception as e:
            print(f"Python engine failed with encoding={encoding}: {type(e).__name__}: {e}")

    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            rows = []
            header = None
            with open(filepath, "r", encoding=encoding, errors="replace", newline="") as f:
                reader = csv.reader(f)
                for i, row in enumerate(reader):
                    if i == 0:
                        header = row
                        continue
                    if len(row) == len(header):
                        rows.append(row)
            if header is not None and len(rows) > 0:
                df = pd.DataFrame(rows, columns=header)
                print(f"Loaded {filepath.name} with encoding={encoding} (manual csv.reader, malformed rows dropped), shape={df.shape}")
                return df
        except Exception as e:
            print(f"Manual csv.reader failed with encoding={encoding}: {type(e).__name__}: {e}")

    print(f"Failed to load {filepath} with any method")
    return None

task2_raw = {}
task2_raw["mcq_train"] = load_jsonl(TASK2_PATHS["mcq_train"])
task2_raw["mcq_val"] = load_jsonl(TASK2_PATHS["mcq_val"])
task2_raw["qa_train"] = load_jsonl(TASK2_PATHS["qa_train"])
task2_raw["qa_val"] = load_jsonl(TASK2_PATHS["qa_val"])
task2_raw["qna_train_csv"] = load_csv_flexible(TASK2_PATHS["qna_train_csv"])
task2_raw["videos"] = load_jsonl(TASK2_PATHS["videos"])
task2_raw["qna_test"] = load_csv_flexible(TASK2_PATHS["qna_test"])
task2_raw["mcq_test"] = load_jsonl(TASK2_PATHS["mcq_test"])
task2_raw["qa_test"] = load_jsonl(TASK2_PATHS["qa_test"])

for key, df in task2_raw.items():
    if df is not None:
        print(f"{key}: shape={df.shape}, columns={list(df.columns)}")
    else:
        print(f"{key}: not loaded")

C engine failed with encoding=utf-8: ParserError: Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.

C engine failed with encoding=utf-8-sig: UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa3 in position 3789: invalid start byte
C engine failed with encoding=latin-1: ParserError: Error tokenizing data. C error: Buffer overflow caught - possible malformed input file.

C engine failed with encoding=cp1252: UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 210938: character maps to <undefined>
Python engine failed with encoding=utf-8: UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa3 in position 3789: invalid start byte
Loaded QnA-TRAIN.csv with encoding=latin-1 (python engine, bad lines skipped), shape=(25344, 5)
Loaded CRYPTO_QnA_TEST.csv with encoding=utf-8 (C engine), shape=(6323, 4)
mcq_train: shape=(1000, 5), columns=['id', 'question', 'options', 'answer_index', 'video_id']
mcq_val: shape=(250, 5), columns=['id

In [ ]:
TEXT_LIKE_COLS = [
    "text", "question", "answer", "context", "chunk", "transcript", "options",
    "choices", "reference_answer", "gold_answer", "comment", "comment_body",
    "title", "selftext", "MAIN", "body", "content", "prompt", "raw_transcript"
]

def clean_text_columns(df, candidate_cols):
    if df is None:
        return None
    df = df.copy()
    for col in df.columns:
        if col in candidate_cols and df[col].dtype == object:
            df[col] = df[col].apply(lambda x: clean_text(x) if isinstance(x, str) else x)

    hashable_cols = [
        col for col in df.columns
        if not df[col].apply(lambda x: isinstance(x, (list, dict))).any()
    ]

    if len(hashable_cols) == len(df.columns):
        df = df.drop_duplicates().reset_index(drop=True)
    else:
        df = df.drop_duplicates(subset=hashable_cols).reset_index(drop=True)

    return df

for key in task2_raw:
    task2_raw[key] = clean_text_columns(task2_raw[key], TEXT_LIKE_COLS)

print("Cleaning complete for all loaded Task 2 dataframes")
for key, df in task2_raw.items():
    if df is not None:
        print(f"{key}: {df.shape}")

Cleaning complete for all loaded Task 2 dataframes
mcq_train: (1000, 5)
mcq_val: (250, 5)
qa_train: (1000, 4)
qa_val: (250, 4)
qna_train_csv: (24886, 5)
videos: (974, 3)
qna_test: (6278, 4)
mcq_test: (500, 4)
qa_test: (500, 3)


In [ ]:
qna_train_df = task2_raw["qna_train_csv"]

if qna_train_df is None:
    raise ValueError("qna_train_csv failed to load. Check file path and integrity at " + str(TASK2_PATHS["qna_train_csv"]))

def build_relevance_dataset(df):
    df = df.copy()
    df = df.rename(columns={"comment_body": "text", "relevance": "relevance_label"})

    df["text"] = df["text"].apply(lambda x: clean_text(x) if isinstance(x, str) else "")
    df = df[df["text"].apply(lambda x: not is_empty_text(x))].reset_index(drop=True)
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

    def normalize_relevance(value):
        if pd.isna(value):
            return np.nan
        try:
            code = int(float(value))
        except (ValueError, TypeError):
            return np.nan
        if code == 1:
            return "RELEVANT"
        if code == 0:
            return "NOT_RELEVANT"
        return np.nan

    df["relevance_label"] = df["relevance_label"].apply(normalize_relevance)
    return df

task2_relevance_df = build_relevance_dataset(qna_train_df)
print("Relevance dataset shape:", task2_relevance_df.shape)
print(task2_relevance_df["relevance_label"].value_counts(dropna=False))
print(task2_relevance_df[["text", "relevance_label"]].head(3))

Relevance dataset shape: (23480, 5)
relevance_label
NOT_RELEVANT    20374
RELEVANT         3070
NaN                36
Name: count, dtype: int64
                                                text relevance_label
0  I have nothing useful to add, but your last pa...    NOT_RELEVANT
1              Hopefully it comes to your area soon!    NOT_RELEVANT
2      To buy coins which arent on the big exchanges    NOT_RELEVANT


In [ ]:
qna_test_df = task2_raw["qna_test"]

def build_relevance_test_dataset(df):
    df = df.copy()
    df = df.rename(columns={"comment_body": "text"})
    df["text"] = df["text"].apply(lambda x: clean_text(x) if isinstance(x, str) else "")
    df["record_id"] = range(len(df))
    return df

if qna_test_df is not None:
    reddit_test_df = build_relevance_test_dataset(qna_test_df)
    print("Reddit relevance test shape:", reddit_test_df.shape)
    print(reddit_test_df[["record_id", "text"]].head(3))
else:
    reddit_test_df = None
    print("qna_test not loaded, reddit_test_df set to None")

Reddit relevance test shape: (6278, 5)
   record_id                                               text
0          0  Wow thanks a lot for this. I didn't like block...
1          1                      The roller coaster continues.
2          2                                  before cum rocket


In [ ]:
plt.figure(figsize=(8, 6))
relevance_counts = task2_relevance_df["relevance_label"].value_counts(dropna=False)
sns.barplot(x=relevance_counts.index.astype(str), y=relevance_counts.values)
plt.title("Reddit relevance label distribution")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "task2_relevance_distribution.png")
plt.close()
print(relevance_counts)

task2_relevance_df["char_length"] = task2_relevance_df["text"].apply(len)
task2_relevance_df["word_count"] = task2_relevance_df["text"].apply(lambda x: len(x.split()))
print("Relevance avg char length:", task2_relevance_df["char_length"].mean())
print("Relevance avg word count:", task2_relevance_df["word_count"].mean())

relevance_label
NOT_RELEVANT    20374
RELEVANT         3070
NaN                36
Name: count, dtype: int64
Relevance avg char length: 163.05170357751277
Relevance avg word count: 27.95736797274276


In [ ]:
relevance_label_encoder = LabelEncoder()
relevance_valid_df = task2_relevance_df[task2_relevance_df["relevance_label"].notna()].reset_index(drop=True)

if len(relevance_valid_df) == 0:
    raise ValueError("No valid relevance labels found after normalization. Check the 'relevance' column values in QnA-TRAIN.csv.")

relevance_label_encoder.fit(relevance_valid_df["relevance_label"])
print("Relevance classes:", list(relevance_label_encoder.classes_))

relevance_train_df, relevance_val_df = train_test_split(
    relevance_valid_df, test_size=0.15, random_state=GLOBAL_SEED, stratify=relevance_valid_df["relevance_label"]
)
relevance_train_df = relevance_train_df.reset_index(drop=True)
relevance_val_df = relevance_val_df.reset_index(drop=True)
print("Relevance train:", len(relevance_train_df), "val:", len(relevance_val_df))

Relevance classes: ['NOT_RELEVANT', 'RELEVANT']
Relevance train: 19927 val: 3517


In [ ]:
class Task1Dataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {"text": self.texts[idx], "label": int(self.labels[idx])}

class DynamicPaddingCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        texts = [item["text"] for item in batch]
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        encoded = self.tokenizer(
            texts, padding=True, truncation=True, max_length=self.max_length, return_tensors="pt"
        )
        encoded["labels"] = labels
        return encoded

class LengthBucketSampler(torch.utils.data.Sampler):
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = lengths
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        indices = np.argsort(self.lengths)
        buckets = [indices[i:i + self.batch_size * 50] for i in range(0, len(indices), self.batch_size * 50)]
        batches = []
        for bucket in buckets:
            bucket = bucket.copy()
            if self.shuffle:
                np.random.shuffle(bucket)
            for i in range(0, len(bucket), self.batch_size):
                batches.append(bucket[i:i + self.batch_size].tolist())
        if self.shuffle:
            np.random.shuffle(batches)
        for batch in batches:
            for idx in batch:
                yield idx

    def __len__(self):
        return len(self.lengths)

RELEVANCE_MODEL_ZOO = [
    "microsoft/deberta-v3-base",
    "roberta-base",
    "microsoft/MiniLM-L12-H384-uncased"
]

def build_relevance_dataloaders(train_df, val_df, tokenizer, config):
    y_train = relevance_label_encoder.transform(train_df["relevance_label"])
    y_val = relevance_label_encoder.transform(val_df["relevance_label"])

    train_ds = Task1Dataset(train_df["text"].tolist(), y_train, tokenizer, config.max_seq_length)
    val_ds = Task1Dataset(val_df["text"].tolist(), y_val, tokenizer, config.max_seq_length)

    collator = DynamicPaddingCollator(tokenizer, config.max_seq_length)
    train_lengths = train_df["word_count"].values
    train_sampler = LengthBucketSampler(train_lengths, config.per_device_batch_size, shuffle=True)

    train_loader = DataLoader(
        train_ds, batch_size=config.per_device_batch_size, sampler=train_sampler,
        collate_fn=collator, num_workers=config.num_workers, pin_memory=config.pin_memory,
        persistent_workers=config.persistent_workers if config.num_workers > 0 else False,
        prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None
    )
    val_loader = DataLoader(
        val_ds, batch_size=config.per_device_batch_size, shuffle=False,
        collate_fn=collator, num_workers=config.num_workers, pin_memory=config.pin_memory,
        persistent_workers=config.persistent_workers if config.num_workers > 0 else False,
        prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None
    )
    return train_loader, val_loader

def sanitize_name(name):
    return name.replace("/", "_").replace("-", "_")

In [ ]:
from tqdm.auto import tqdm

def evaluate_model(model, val_loader, device, num_classes):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False, mininterval=2.0):
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                outputs = model(**batch)
                logits = outputs.logits
            probs = F.softmax(logits.float(), dim=-1)
            preds = torch.argmax(probs, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, macro_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    _, _, weighted_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": precision, "recall": recall, "macro_f1": macro_f1, "weighted_f1": weighted_f1}, all_preds, all_labels, all_probs

def train_relevance_model(model_name, config, save_dir):
    save_dir.mkdir(parents=True, exist_ok=True)
    result = {"model_name": model_name, "status": "failed"}
    tokenizer = None
    model = None
    try:
        print(f"[{model_name}] Loading tokenizer...", flush=True)
        num_classes = len(relevance_label_encoder.classes_)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        print(f"[{model_name}] Building dataloaders...", flush=True)
        train_loader, val_loader = build_relevance_dataloaders(relevance_train_df, relevance_val_df, tokenizer, config)
        print(f"[{model_name}] train_batches={len(train_loader)} val_batches={len(val_loader)}", flush=True)

        model_config = AutoConfig.from_pretrained(model_name, num_labels=num_classes)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, config=model_config)
        if config.use_gradient_checkpointing:
            try:
                model.gradient_checkpointing_enable()
            except Exception:
                pass
        model.to(DEVICE)
        print(f"[{model_name}] Model loaded on {DEVICE}. Starting training...", flush=True)

        y_train = relevance_label_encoder.transform(relevance_train_df["relevance_label"])
        counts = np.bincount(y_train, minlength=num_classes)
        counts = np.where(counts == 0, 1, counts)
        weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float).to(DEVICE)
        loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=config.label_smoothing)

        optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        total_steps = (len(train_loader) // config.gradient_accumulation_steps) * config.num_epochs
        warmup_steps = int(total_steps * config.warmup_ratio)
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        scaler = GradScaler(enabled=HW.use_amp and HW.amp_dtype == torch.float16)

        best_macro_f1 = -1
        patience_counter = 0
        start_time = time.time()

        for epoch in range(config.num_epochs):
            model.train()
            optimizer.zero_grad()
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}", mininterval=2.0)
            for step, batch in enumerate(progress_bar):
                labels = batch.pop("labels").to(DEVICE)
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                try:
                    with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                        outputs = model(**batch)
                        loss = loss_fn(outputs.logits, labels) / config.gradient_accumulation_steps
                    if scaler.is_enabled():
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        optimizer.zero_grad()
                        clear_memory()
                        continue
                    else:
                        raise

                progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

                if (step + 1) % config.gradient_accumulation_steps == 0:
                    if scaler.is_enabled():
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                        optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            val_metrics, val_preds, val_labels, val_probs = evaluate_model(model, val_loader, DEVICE, num_classes)
            print(f"[{model_name}] Epoch {epoch+1} macro_f1={val_metrics['macro_f1']:.4f}", flush=True)

            if val_metrics["macro_f1"] > best_macro_f1:
                best_macro_f1 = val_metrics["macro_f1"]
                patience_counter = 0
                model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)
                roc_auc = None
                ap = None
                try:
                    probs_arr = np.array(val_probs)
                    roc_auc = roc_auc_score(val_labels, probs_arr[:, 1])
                    ap = average_precision_score(val_labels, probs_arr[:, 1])
                except Exception:
                    pass
                with open(save_dir / "best_metrics.json", "w") as f:
                    json.dump({**val_metrics, "roc_auc": roc_auc, "average_precision": ap}, f, indent=2)
            else:
                patience_counter += 1
                if patience_counter >= config.early_stopping_patience:
                    print(f"Early stopping at epoch {epoch+1}", flush=True)
                    break

            clear_memory()

        training_time = time.time() - start_time
        result.update({
            "status": "success",
            "best_macro_f1": best_macro_f1,
            "training_time_sec": training_time,
            "save_dir": str(save_dir),
            "gpu_stats": get_gpu_memory_stats()
        })
    except Exception as e:
        print(f"Relevance training failed for {model_name}: {e}", flush=True)
        traceback.print_exc()
        result["error"] = str(e)
    finally:
        delete_model_safely(model, tokenizer)

    return result

relevance_config = TrainingConfig(
    max_seq_length=128, per_device_batch_size=16, gradient_accumulation_steps=2,
    num_epochs=3, num_workers=0, persistent_workers=False, prefetch_factor=None
)

relevance_experiments = []
for model_name in RELEVANCE_MODEL_ZOO:
    save_dir = CHECKPOINT_DIR / "task2_relevance" / sanitize_name(model_name)
    result = safe_execute(
        train_relevance_model, model_name, relevance_config, save_dir,
        default_return={"model_name": model_name, "status": "failed"},
        log_prefix=f"relevance_{model_name}"
    )
    relevance_experiments.append(result)
    clear_memory()

with open(METRICS_DIR / "task2_relevance_experiments.json", "w") as f:
    json.dump(relevance_experiments, f, indent=2, default=str)

relevance_success = [e for e in relevance_experiments if e["status"] == "success"]
relevance_success.sort(key=lambda x: x["best_macro_f1"], reverse=True)
print("Relevance model ranking:")
for e in relevance_success:
    print(e["model_name"], e["best_macro_f1"])

task2_best_relevance_model = relevance_success[0] if relevance_success else None

[microsoft/deberta-v3-base] Loading tokenizer...
[microsoft/deberta-v3-base] Building dataloaders...
[microsoft/deberta-v3-base] train_batches=1246 val_batches=220


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[microsoft/deberta-v3-base] Model loaded on cuda. Starting training...


Epoch 1/3:   0%|          | 0/1246 [00:00<?, ?it/s]

Validating:   0%|          | 0/220 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/220 [00:00<?, ?it/s]

[microsoft/MiniLM-L12-H384-uncased] Epoch 2 macro_f1=0.6248


Epoch 3/3:   0%|          | 0/1246 [00:00<?, ?it/s]

Validating:   0%|          | 0/220 [00:00<?, ?it/s]

[microsoft/MiniLM-L12-H384-uncased] Epoch 3 macro_f1=0.6030
Relevance model ranking:
microsoft/MiniLM-L12-H384-uncased 0.6248100306508562
roberta-base 0.5938268107731195
microsoft/deberta-v3-base 0.5849163994513877


In [ ]:
def run_inference_batches(model, tokenizer, texts, max_length, device, batch_size=32):
    def infer_batch(bs):
        results = []
        for i in range(0, len(texts), bs):
            chunk = texts[i:i + bs]
            encoded = tokenizer(chunk, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            encoded = {k: v.to(device) for k, v in encoded.items()}
            with torch.no_grad():
                with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                    outputs = model(**encoded)
                    logits = outputs.logits
            probs = F.softmax(logits.float(), dim=-1).cpu().numpy()
            results.append(probs)
        return np.concatenate(results, axis=0)
    probs, _ = OOM_RUNNER.run(infer_batch, batch_size)
    return probs

def load_checkpoint_for_inference(save_dir):
    tokenizer = AutoTokenizer.from_pretrained(save_dir)
    model = AutoModelForSequenceClassification.from_pretrained(save_dir)
    model.to(DEVICE)
    model.eval()
    return model, tokenizer

def predict_relevance_for_texts(texts, best_model_info):
    if best_model_info is None:
        return None, None
    save_dir = Path(best_model_info["save_dir"])
    model = None
    tokenizer = None
    try:
        model, tokenizer = load_checkpoint_for_inference(save_dir)
        probs = run_inference_batches(model, tokenizer, texts, relevance_config.max_seq_length, DEVICE)
        preds = np.argmax(probs, axis=1)
        labels_text = relevance_label_encoder.inverse_transform(preds)
        return labels_text, probs
    except Exception as e:
        print(f"Relevance test prediction failed: {e}")
        traceback.print_exc()
        return None, None
    finally:
        delete_model_safely(model, tokenizer)

reddit_predictions = {}
if reddit_test_df is not None and task2_best_relevance_model is not None:
    labels_text, probs = safe_execute(
        predict_relevance_for_texts, reddit_test_df["text"].tolist(), task2_best_relevance_model,
        default_return=(None, None), log_prefix="reddit_relevance_test"
    )
    if labels_text is not None:
        reddit_predictions["submission_1"] = pd.DataFrame({"id": reddit_test_df["record_id"], "relevance": labels_text})
        print("Reddit relevance predictions generated:", reddit_predictions["submission_1"].shape)
clear_memory()

Reddit relevance predictions generated: (6278, 2)


In [ ]:
videos_df = task2_raw.get("videos")

def get_video_transcript_map():
    mapping = {}
    if videos_df is not None:
        for _, row in videos_df.iterrows():
            mapping[row["video_id"]] = row.get("raw_transcript", "")
    return mapping

video_transcript_map = get_video_transcript_map()

def build_qa_dataset(df):
    df = df.copy()
    df["question"] = df["question"].apply(lambda x: clean_text(x) if isinstance(x, str) else "")
    df["answer"] = df["answer"].apply(lambda x: clean_text(x) if isinstance(x, str) else "")
    df["context"] = df["video_id"].apply(lambda vid: clean_text(str(video_transcript_map.get(vid, ""))))

    df = df[df["question"].apply(lambda x: not is_empty_text(x))].reset_index(drop=True)
    df = df[df["answer"].apply(lambda x: not is_empty_text(x))].reset_index(drop=True)
    df = df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)
    return df

qa_train_df = build_qa_dataset(task2_raw["qa_train"])
qa_val_df = build_qa_dataset(task2_raw["qa_val"])

print("QA train shape:", qa_train_df.shape)
print("QA val shape:", qa_val_df.shape)
print(qa_train_df[["question", "answer"]].head(2))

QA train shape: (1000, 5)
QA val shape: (250, 5)
                                            question  \
0  How does Bitcoin compare to gold in some narra...   
1  Why might Off the Grid's design choice of oper...   

                                              answer  
0  Bitcoin is seen as digital gold and an alterna...  
1  Off the Grid operates on its own subnet to ens...  


In [ ]:
qa_train_df["question_word_count"] = qa_train_df["question"].apply(lambda x: len(x.split()))
qa_train_df["context_word_count"] = qa_train_df["context"].apply(lambda x: len(x.split()))
qa_train_df["answer_word_count"] = qa_train_df["answer"].apply(lambda x: len(x.split()))

print("Question avg words:", qa_train_df["question_word_count"].mean())
print("Context avg words:", qa_train_df["context_word_count"].mean())
print("Answer avg words:", qa_train_df["answer_word_count"].mean())

def plot_length_histogram(df, column, title, save_path, bins=50):
    plt.figure(figsize=(10, 6))
    plt.hist(df[column], bins=bins, edgecolor="black")
    plt.title(title)
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

plot_length_histogram(qa_train_df, "context_word_count", "QA context word count", PLOTS_DIR / "task2_qa_context_length.png")
plot_length_histogram(qa_train_df, "answer_word_count", "QA answer word count", PLOTS_DIR / "task2_qa_answer_length.png")

Question avg words: 16.553
Context avg words: 4105.811
Answer avg words: 25.254


In [ ]:
class Seq2SeqQADataset(Dataset):
    def __init__(self, df, tokenizer, max_input_length=384, max_target_length=64):
        self.questions = df["question"].tolist()
        self.contexts = df["context"].tolist()
        self.answers = df["answer"].tolist()
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        input_text = f"question: {self.questions[idx]} context: {self.contexts[idx]}"
        target_text = self.answers[idx]
        return {"input_text": input_text, "target_text": target_text}

class Seq2SeqQACollator:
    def __init__(self, tokenizer, max_input_length=384, max_target_length=64):
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __call__(self, batch):
        inputs = [item["input_text"] for item in batch]
        targets = [item["target_text"] for item in batch]
        model_inputs = self.tokenizer(
            inputs, padding=True, truncation=True, max_length=self.max_input_length, return_tensors="pt"
        )
        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                targets, padding=True, truncation=True, max_length=self.max_target_length, return_tensors="pt"
            )
        label_ids = labels["input_ids"]
        label_ids[label_ids == self.tokenizer.pad_token_id] = -100
        model_inputs["labels"] = label_ids
        return model_inputs

In [ ]:
import evaluate as hf_evaluate

rouge_metric = hf_evaluate.load("rouge")
bleu_metric = hf_evaluate.load("sacrebleu")

def normalize_answer_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_exact_match(preds, refs):
    matches = [int(normalize_answer_text(p) == normalize_answer_text(r)) for p, r in zip(preds, refs)]
    return float(np.mean(matches)) if matches else 0.0

def compute_token_f1(preds, refs):
    f1_scores = []
    for p, r in zip(preds, refs):
        p_tokens = normalize_answer_text(p).split()
        r_tokens = normalize_answer_text(r).split()
        common = Counter(p_tokens) & Counter(r_tokens)
        num_same = sum(common.values())
        if num_same == 0:
            f1_scores.append(0.0)
            continue
        precision = num_same / len(p_tokens) if p_tokens else 0.0
        recall = num_same / len(r_tokens) if r_tokens else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)
    return float(np.mean(f1_scores)) if f1_scores else 0.0

def compute_bertscore(preds, refs):
    try:
        bertscore_metric = hf_evaluate.load("bertscore")
        results = bertscore_metric.compute(predictions=preds, references=refs, lang="en", model_type="distilbert-base-uncased")
        return float(np.mean(results["f1"]))
    except Exception as e:
        print(f"BERTScore computation failed: {e}")
        return None

def compute_qa_metrics(preds, refs):
    metrics = {}
    metrics["exact_match"] = compute_exact_match(preds, refs)
    metrics["token_f1"] = compute_token_f1(preds, refs)
    try:
        rouge_result = rouge_metric.compute(predictions=preds, references=refs)
        metrics["rougeL"] = float(rouge_result["rougeL"])
        metrics["rouge1"] = float(rouge_result["rouge1"])
    except Exception as e:
        print(f"ROUGE computation failed: {e}")
        metrics["rougeL"] = None
        metrics["rouge1"] = None
    try:
        bleu_refs = [[r] for r in refs]
        bleu_result = bleu_metric.compute(predictions=preds, references=bleu_refs)
        metrics["bleu"] = float(bleu_result["score"])
    except Exception as e:
        print(f"BLEU computation failed: {e}")
        metrics["bleu"] = None
    metrics["bertscore_f1"] = compute_bertscore(preds, refs)
    return metrics

In [ ]:
def generate_predictions(model, tokenizer, dataloader, device, max_target_length=64):
    model.eval()
    all_preds = []
    all_refs = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Generating", leave=False, mininterval=2.0):
            labels = batch.pop("labels")
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                generated = model.generate(
                    input_ids=input_ids, attention_mask=attention_mask,
                    max_length=max_target_length, num_beams=4
                )
            decoded_preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
            labels_clean = labels.clone()
            labels_clean[labels_clean == -100] = tokenizer.pad_token_id
            decoded_refs = tokenizer.batch_decode(labels_clean, skip_special_tokens=True)
            all_preds.extend(decoded_preds)
            all_refs.extend(decoded_refs)
    return all_preds, all_refs

def train_qa_model(model_name, config, save_dir, max_input_length=384, max_target_length=64):
    save_dir.mkdir(parents=True, exist_ok=True)
    result = {"model_name": model_name, "status": "failed"}
    tokenizer = None
    model = None
    try:
        print(f"[{model_name}] Loading tokenizer and model...", flush=True)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        if config.use_gradient_checkpointing:
            try:
                model.gradient_checkpointing_enable()
            except Exception:
                pass
        model.to(DEVICE)

        train_ds = Seq2SeqQADataset(qa_train_df, tokenizer, max_input_length, max_target_length)
        val_ds = Seq2SeqQADataset(qa_val_df, tokenizer, max_input_length, max_target_length)
        collator = Seq2SeqQACollator(tokenizer, max_input_length, max_target_length)

        train_loader = DataLoader(
            train_ds, batch_size=config.per_device_batch_size, shuffle=True, collate_fn=collator,
            num_workers=config.num_workers, pin_memory=config.pin_memory,
            persistent_workers=config.persistent_workers if config.num_workers > 0 else False,
            prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None
        )
        val_loader = DataLoader(
            val_ds, batch_size=config.per_device_batch_size, shuffle=False, collate_fn=collator,
            num_workers=config.num_workers, pin_memory=config.pin_memory,
            persistent_workers=config.persistent_workers if config.num_workers > 0 else False,
            prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None
        )
        print(f"[{model_name}] train_batches={len(train_loader)} val_batches={len(val_loader)}", flush=True)

        optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        total_steps = (len(train_loader) // config.gradient_accumulation_steps) * config.num_epochs
        warmup_steps = int(total_steps * config.warmup_ratio)
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        scaler = GradScaler(enabled=HW.use_amp and HW.amp_dtype == torch.float16)

        best_rougeL = -1
        patience_counter = 0
        start_time = time.time()

        for epoch in range(config.num_epochs):
            model.train()
            optimizer.zero_grad()
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}", mininterval=2.0)
            for step, batch in enumerate(progress_bar):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                try:
                    with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                        outputs = model(**batch)
                        loss = outputs.loss / config.gradient_accumulation_steps
                    if scaler.is_enabled():
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()

                    if (step + 1) % config.gradient_accumulation_steps == 0:
                        if scaler.is_enabled():
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                            optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad()

                    progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print(f"OOM at step {step}, clearing memory and skipping batch", flush=True)
                        optimizer.zero_grad()
                        clear_memory()
                        continue
                    else:
                        raise

            preds, refs = generate_predictions(model, tokenizer, val_loader, DEVICE, max_target_length)
            metrics = compute_qa_metrics(preds, refs)
            print(f"[{model_name}] Epoch {epoch+1} rougeL={metrics['rougeL']} EM={metrics['exact_match']} F1={metrics['token_f1']}", flush=True)

            current_score = metrics["rougeL"] if metrics["rougeL"] is not None else metrics["token_f1"]
            if current_score > best_rougeL:
                best_rougeL = current_score
                patience_counter = 0
                model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)
                with open(save_dir / "best_metrics.json", "w") as f:
                    json.dump(metrics, f, indent=2)
            else:
                patience_counter += 1
                if patience_counter >= config.early_stopping_patience:
                    print(f"Early stopping at epoch {epoch+1}", flush=True)
                    break

            clear_memory()

        training_time = time.time() - start_time
        result.update({
            "status": "success", "best_rougeL": best_rougeL, "training_time_sec": training_time,
            "save_dir": str(save_dir), "gpu_stats": get_gpu_memory_stats()
        })
    except Exception as e:
        print(f"QA training failed for {model_name}: {e}", flush=True)
        traceback.print_exc()
        result["error"] = str(e)
    finally:
        delete_model_safely(model, tokenizer)

    return result

QA_MODEL_ZOO = ["google/flan-t5-base"]

qa_config = TrainingConfig(
    learning_rate=3e-4, per_device_batch_size=8, gradient_accumulation_steps=4,
    num_epochs=3, max_seq_length=384, num_workers=0, persistent_workers=False, prefetch_factor=None
)

qa_experiments = []
for model_name in QA_MODEL_ZOO:
    save_dir = CHECKPOINT_DIR / "task2_qa" / sanitize_name(model_name)
    result = safe_execute(
        train_qa_model, model_name, qa_config, save_dir,
        default_return={"model_name": model_name, "status": "failed"},
        log_prefix=f"qa_{model_name}"
    )
    qa_experiments.append(result)
    clear_memory()

with open(METRICS_DIR / "task2_qa_experiments.json", "w") as f:
    json.dump(qa_experiments, f, indent=2, default=str)

qa_success = [e for e in qa_experiments if e["status"] == "success"]
qa_success.sort(key=lambda x: x["best_rougeL"], reverse=True)
print("QA model ranking:")
for e in qa_success:
    print(e["model_name"], e["best_rougeL"])

task2_best_qa_model = qa_success[0] if qa_success else None

[google/flan-t5-base] Loading tokenizer and model...
[google/flan-t5-base] train_batches=125 val_batches=32


Epoch 1/3:   0%|          | 0/125 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Generating:   0%|          | 0/32 [00:00<?, ?it/s]

[google/flan-t5-base] Epoch 1 rougeL=0.0 EM=0.0 F1=0.0


...

Generating:   0%|          | 0/32 [00:00<?, ?it/s]

[google/flan-t5-base] Epoch 2 rougeL=0.0 EM=0.0 F1=0.0


Epoch 3/3:   0%|          | 0/125 [00:00<?, ?it/s]

Generating:   0%|          | 0/32 [00:00<?, ?it/s]

[google/flan-t5-base] Epoch 3 rougeL=0.0 EM=0.0 F1=0.0
Early stopping at epoch 3
QA model ranking:
google/flan-t5-base 0.0


In [ ]:
mcq_train_raw = task2_raw["mcq_train"]
mcq_val_raw = task2_raw["mcq_val"]

def parse_options_field(value):
    if isinstance(value, list):
        return [clean_text(str(v)) for v in value]
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, list):
                return [clean_text(str(v)) for v in parsed]
        except Exception:
            pass
        parts = re.split(r"\s*[\|;]\s*", value)
        if len(parts) > 1:
            return [clean_text(p) for p in parts]
        return [clean_text(value)]
    return []

def safe_notna(value):
    if isinstance(value, (list, dict, np.ndarray)):
        return True
    return pd.notna(value)

def build_mcq_dataset(df):
    df = df.copy()
    records = []
    for _, row in df.iterrows():
        question_val = row.get("question")
        question = clean_text(str(question_val)) if safe_notna(question_val) else ""

        context = clean_text(str(video_transcript_map.get(row.get("video_id"), "")))

        options_val = row.get("options")
        options = parse_options_field(options_val) if safe_notna(options_val) else []

        answer_index_val = row.get("answer_index")
        answer_idx = int(answer_index_val) if safe_notna(answer_index_val) else -1

        if is_empty_text(question) or len(options) < 2 or answer_idx < 0 or answer_idx >= len(options):
            continue

        records.append({
            "record_id": row.get("id"),
            "question": question,
            "context": context,
            "options": options,
            "answer_idx": answer_idx
        })
    return pd.DataFrame(records)

mcq_train_df = build_mcq_dataset(mcq_train_raw)
mcq_val_df = build_mcq_dataset(mcq_val_raw)

print("MCQ train shape:", mcq_train_df.shape)
print("MCQ val shape:", mcq_val_df.shape)
mcq_train_df["num_options"] = mcq_train_df["options"].apply(len)
print("Options count distribution:", mcq_train_df["num_options"].value_counts())
max_num_choices = int(mcq_train_df["num_options"].max())
print("Max number of choices:", max_num_choices)

MCQ train shape: (1000, 5)
MCQ val shape: (250, 5)
Options count distribution: num_options
4    1000
Name: count, dtype: int64
Max number of choices: 4


In [ ]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256, max_choices=4):
        self.questions = df["question"].tolist()
        self.contexts = df["context"].tolist()
        self.options = df["options"].tolist()
        self.answers = df["answer_idx"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_choices = max_choices

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        options = list(self.options[idx][:self.max_choices])
        while len(options) < self.max_choices:
            options.append("")
        first_sentences = [f"{self.contexts[idx]} {self.questions[idx]}".strip()] * self.max_choices
        second_sentences = options
        return {"first_sentences": first_sentences, "second_sentences": second_sentences, "label": self.answers[idx]}

class MCQCollator:
    def __init__(self, tokenizer, max_length=256):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        batch_size = len(batch)
        num_choices = len(batch[0]["first_sentences"])
        first_sentences = sum([item["first_sentences"] for item in batch], [])
        second_sentences = sum([item["second_sentences"] for item in batch], [])
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)

        encoded = self.tokenizer(
            first_sentences, second_sentences, padding=True, truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        encoded = {k: v.view(batch_size, num_choices, -1) for k, v in encoded.items()}
        encoded["labels"] = labels
        return encoded

def evaluate_mcq_model(model, val_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating MCQ", leave=False, mininterval=2.0):
            labels = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                outputs = model(**batch)
                logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    _, _, macro_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_f1": macro_f1}, all_preds, all_labels

def train_mcq_model(model_name, config, save_dir, max_choices):
    save_dir.mkdir(parents=True, exist_ok=True)
    result = {"model_name": model_name, "status": "failed"}
    tokenizer = None
    model = None
    try:
        print(f"[{model_name}] Loading tokenizer and model...", flush=True)
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForMultipleChoice.from_pretrained(model_name)
        if config.use_gradient_checkpointing:
            try:
                model.gradient_checkpointing_enable()
            except Exception:
                pass
        model.to(DEVICE)

        train_ds = MCQDataset(mcq_train_df, tokenizer, config.max_seq_length, max_choices)
        val_ds = MCQDataset(mcq_val_df, tokenizer, config.max_seq_length, max_choices)
        collator = MCQCollator(tokenizer, config.max_seq_length)

        train_loader = DataLoader(
            train_ds, batch_size=config.per_device_batch_size, shuffle=True, collate_fn=collator,
            num_workers=config.num_workers, pin_memory=config.pin_memory,
            persistent_workers=config.persistent_workers if config.num_workers > 0 else False,
            prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None
        )
        val_loader = DataLoader(
            val_ds, batch_size=config.per_device_batch_size, shuffle=False, collate_fn=collator,
            num_workers=config.num_workers, pin_memory=config.pin_memory,
            persistent_workers=config.persistent_workers if config.num_workers > 0 else False,
            prefetch_factor=config.prefetch_factor if config.num_workers > 0 else None
        )
        print(f"[{model_name}] train_batches={len(train_loader)} val_batches={len(val_loader)}", flush=True)

        loss_fn = nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
        total_steps = (len(train_loader) // config.gradient_accumulation_steps) * config.num_epochs
        warmup_steps = int(total_steps * config.warmup_ratio)
        scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        scaler = GradScaler(enabled=HW.use_amp and HW.amp_dtype == torch.float16)

        best_macro_f1 = -1
        patience_counter = 0
        start_time = time.time()

        for epoch in range(config.num_epochs):
            model.train()
            optimizer.zero_grad()
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}", mininterval=2.0)
            for step, batch in enumerate(progress_bar):
                labels = batch.pop("labels").to(DEVICE)
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                try:
                    with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                        outputs = model(**batch)
                        loss = loss_fn(outputs.logits, labels) / config.gradient_accumulation_steps
                    if scaler.is_enabled():
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        optimizer.zero_grad()
                        clear_memory()
                        continue
                    else:
                        raise

                progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

                if (step + 1) % config.gradient_accumulation_steps == 0:
                    if scaler.is_enabled():
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
                        optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            val_metrics, val_preds, val_labels = evaluate_mcq_model(model, val_loader, DEVICE)
            print(f"[{model_name}] Epoch {epoch+1} macro_f1={val_metrics['macro_f1']:.4f} acc={val_metrics['accuracy']:.4f}", flush=True)

            if val_metrics["macro_f1"] > best_macro_f1:
                best_macro_f1 = val_metrics["macro_f1"]
                patience_counter = 0
                model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)
                cm = confusion_matrix(val_labels, val_preds)
                plt.figure(figsize=(6, 5))
                sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
                plt.title(f"MCQ Confusion Matrix - {model_name}")
                plt.savefig(PLOTS_DIR / f"mcq_cm_{sanitize_name(model_name)}.png")
                plt.close()
                with open(save_dir / "best_metrics.json", "w") as f:
                    json.dump(val_metrics, f, indent=2)
            else:
                patience_counter += 1
                if patience_counter >= config.early_stopping_patience:
                    print(f"Early stopping at epoch {epoch+1}", flush=True)
                    break

            clear_memory()

        training_time = time.time() - start_time
        result.update({
            "status": "success",
            "best_macro_f1": best_macro_f1,
            "training_time_sec": training_time,
            "save_dir": str(save_dir),
            "max_choices": max_choices,
            "max_seq_length": config.max_seq_length,
            "gpu_stats": get_gpu_memory_stats()
        })
    except Exception as e:
        print(f"MCQ training failed for {model_name}: {e}", flush=True)
        traceback.print_exc()
        result["error"] = str(e)
    finally:
        delete_model_safely(model, tokenizer)

    return result

MCQ_MODEL_ZOO = ["roberta-base", "microsoft/deberta-v3-base"]

mcq_config = TrainingConfig(
    learning_rate=2e-5, per_device_batch_size=4, gradient_accumulation_steps=4,
    num_epochs=3, max_seq_length=256, num_workers=0, persistent_workers=False, prefetch_factor=None
)

mcq_experiments = []
for model_name in MCQ_MODEL_ZOO:
    save_dir = CHECKPOINT_DIR / "task2_mcq" / sanitize_name(model_name)
    result = safe_execute(
        train_mcq_model, model_name, mcq_config, save_dir, max_num_choices,
        default_return={"model_name": model_name, "status": "failed"},
        log_prefix=f"mcq_{model_name}"
    )
    mcq_experiments.append(result)
    clear_memory()

with open(METRICS_DIR / "task2_mcq_experiments.json", "w") as f:
    json.dump(mcq_experiments, f, indent=2, default=str)

mcq_success = [e for e in mcq_experiments if e["status"] == "success"]
mcq_success.sort(key=lambda x: x["best_macro_f1"], reverse=True)
print("MCQ model ranking:")
for e in mcq_success:
    print(e["model_name"], e["best_macro_f1"])

task2_best_mcq_model = mcq_success[0] if mcq_success else None

[roberta-base] Loading tokenizer and model...


Some weights of RobertaForMultipleChoice were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[roberta-base] train_batches=250 val_batches=63


Epoch 1/3:   0%|          | 0/250 [00:00<?, ?it/s]

Validating MCQ:   0%|          | 0/63 [00:00<?, ?it/s]

...

Validating MCQ:   0%|          | 0/63 [00:00<?, ?it/s]

[microsoft/deberta-v3-base] Epoch 2 macro_f1=0.2082 acc=0.2080


Epoch 3/3:   0%|          | 0/250 [00:00<?, ?it/s]

Validating MCQ:   0%|          | 0/63 [00:00<?, ?it/s]

[microsoft/deberta-v3-base] Epoch 3 macro_f1=0.2082 acc=0.2080
Early stopping at epoch 3
MCQ model ranking:
roberta-base 0.22800055927078716
microsoft/deberta-v3-base 0.20824589647264527


In [ ]:
TRANSCRIPT_CANDIDATES = ["raw_transcript"]

def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start += chunk_size - overlap
    return chunks

video_chunks = []
if videos_df is not None:
    for _, row in videos_df.iterrows():
        transcript = clean_text(str(row.get("raw_transcript", "")))
        video_id = row.get("video_id")
        if is_empty_text(transcript):
            continue
        for i, chunk in enumerate(chunk_text(transcript)):
            video_chunks.append({"video_id": video_id, "chunk_id": i, "chunk_text": chunk})

video_chunks_df = pd.DataFrame(video_chunks)
print("Total chunks generated:", len(video_chunks_df))
if len(video_chunks_df) > 0:
    print(video_chunks_df.head(3))

Total chunks generated: 23679
      video_id  chunk_id                                         chunk_text
0  --swStNGLYA         0  behind my eyes. Got my shoes cracked. Only two...
1  --swStNGLYA         1  having aren't you just so excited. But realist...
2  --swStNGLYA         2  expect now? Some people are freaking out. They...


In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_CANDIDATES = ["BAAI/bge-small-en-v1.5", "intfloat/e5-small-v2"]

def load_embedding_model():
    for model_name in EMBEDDING_MODEL_CANDIDATES:
        try:
            model = SentenceTransformer(model_name, device=str(DEVICE))
            print(f"Loaded embedding model: {model_name}")
            return model, model_name
        except Exception as e:
            print(f"Failed to load {model_name}: {e}")
            continue
    return None, None

embedding_model, embedding_model_name = load_embedding_model()

chunk_embeddings = None
if embedding_model is not None and len(video_chunks_df) > 0:
    def embed_batches(bs):
        return embedding_model.encode(
            video_chunks_df["chunk_text"].tolist(), batch_size=bs,
            show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True
        )
    chunk_embeddings, used_bs = OOM_RUNNER.run(embed_batches, 32)
    print("Chunk embeddings shape:", chunk_embeddings.shape)

clear_memory()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: BAAI/bge-small-en-v1.5


Batches:   0%|          | 0/740 [00:00<?, ?it/s]

Chunk embeddings shape: (23679, 384)


In [ ]:
def retrieve_top_k_chunks(question, k=3):
    if embedding_model is None or chunk_embeddings is None or len(video_chunks_df) == 0:
        return []
    query_emb = embedding_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    scores = np.dot(chunk_embeddings, query_emb.T).squeeze(-1)
    top_idx = np.argsort(scores)[::-1][:k]
    return video_chunks_df.iloc[top_idx]["chunk_text"].tolist()

def build_rag_context(question, k=3):
    return " ".join(retrieve_top_k_chunks(question, k))

In [ ]:
def build_rag_qa_val_set(qa_val_df, k=3):
    rag_df = qa_val_df.copy()
    if len(video_chunks_df) == 0 or embedding_model is None:
        rag_df["context"] = qa_val_df["context"]
        return rag_df
    new_contexts = []
    for question in qa_val_df["question"]:
        rag_context = safe_execute(build_rag_context, question, k, default_return="", log_prefix="rag_context_build")
        new_contexts.append(rag_context if not is_empty_text(rag_context) else "")
    rag_df["context"] = new_contexts
    return rag_df

def evaluate_qa_pipeline(model_save_dir, eval_df, max_input_length=384, max_target_length=64, batch_size=8):
    tokenizer = None
    model = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_save_dir)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_save_dir)
        model.to(DEVICE)
        ds = Seq2SeqQADataset(eval_df, tokenizer, max_input_length, max_target_length)
        collator = Seq2SeqQACollator(tokenizer, max_input_length, max_target_length)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collator)
        preds, refs = generate_predictions(model, tokenizer, loader, DEVICE, max_target_length)
        metrics = compute_qa_metrics(preds, refs)
        return metrics, preds
    except Exception as e:
        print(f"QA pipeline evaluation failed: {e}")
        traceback.print_exc()
        return None, None
    finally:
        delete_model_safely(model, tokenizer)

rag_comparison_results = {}
if task2_best_qa_model is not None:
    baseline_metrics, _ = safe_execute(
        evaluate_qa_pipeline, Path(task2_best_qa_model["save_dir"]), qa_val_df,
        default_return=(None, None), log_prefix="qa_baseline_eval"
    )
    rag_val_df = build_rag_qa_val_set(qa_val_df, k=3)
    rag_metrics, _ = safe_execute(
        evaluate_qa_pipeline, Path(task2_best_qa_model["save_dir"]), rag_val_df,
        default_return=(None, None), log_prefix="qa_rag_eval"
    )
    rag_comparison_results = {"non_rag": baseline_metrics, "rag": rag_metrics}
    print("Non-RAG metrics:", baseline_metrics)
    print("RAG metrics:", rag_metrics)

with open(METRICS_DIR / "task2_rag_comparison.json", "w") as f:
    json.dump(rag_comparison_results, f, indent=2, default=str)

Generating:   0%|          | 0/32 [00:00<?, ?it/s]

Generating:   0%|          | 0/32 [00:00<?, ?it/s]

Non-RAG metrics: {'exact_match': 0.0, 'token_f1': 0.0, 'rougeL': 0.0, 'rouge1': 0.0, 'bleu': 0.0, 'bertscore_f1': 0.5734621886014938}
RAG metrics: {'exact_match': 0.0, 'token_f1': 0.0, 'rougeL': 0.0, 'rouge1': 0.0, 'bleu': 0.0, 'bertscore_f1': 0.5734621886014938}


In [ ]:
leaderboard_rows = []

for e in relevance_success:
    leaderboard_rows.append({
        "task": "reddit_relevance", "model": e["model_name"], "metric_name": "macro_f1",
        "metric_value": e["best_macro_f1"], "training_time_sec": e["training_time_sec"],
        "gpu_memory_gb": e["gpu_stats"]["allocated_gb"]
    })

for e in qa_success:
    leaderboard_rows.append({
        "task": "qa", "model": e["model_name"], "metric_name": "rougeL",
        "metric_value": e["best_rougeL"], "training_time_sec": e["training_time_sec"],
        "gpu_memory_gb": e["gpu_stats"]["allocated_gb"]
    })

for e in mcq_success:
    leaderboard_rows.append({
        "task": "mcq", "model": e["model_name"], "metric_name": "macro_f1",
        "metric_value": e["best_macro_f1"], "training_time_sec": e["training_time_sec"],
        "gpu_memory_gb": e["gpu_stats"]["allocated_gb"]
    })

task2_leaderboard_df = pd.DataFrame(leaderboard_rows).sort_values(["task", "metric_value"], ascending=[True, False])
task2_leaderboard_df.to_csv(METRICS_DIR / "task2_leaderboard.csv", index=False)
print(task2_leaderboard_df.to_string())

               task                              model metric_name  metric_value  training_time_sec  gpu_memory_gb
4               mcq                       roberta-base    macro_f1      0.228001         254.779716          1.940
5               mcq          microsoft/deberta-v3-base    macro_f1      0.208246        1069.673062          2.302
3                qa                google/flan-t5-base      rougeL      0.000000         609.965527          2.184
0  reddit_relevance  microsoft/MiniLM-L12-H384-uncased    macro_f1      0.624810         620.742166          0.435
1  reddit_relevance                       roberta-base    macro_f1      0.593827         647.049080          1.460
2  reddit_relevance          microsoft/deberta-v3-base    macro_f1      0.584916        1589.673851          2.131


In [ ]:
mcq_test_raw = task2_raw.get("mcq_test")
mcq_test_df = pd.DataFrame()

if mcq_test_raw is not None:
    mcq_test_records = []
    for idx, row in mcq_test_raw.iterrows():
        question_val = row.get("question")
        question = clean_text(str(question_val)) if safe_notna(question_val) else ""
        context = clean_text(str(video_transcript_map.get(row.get("video_id"), "")))
        options_val = row.get("options")
        options = parse_options_field(options_val) if safe_notna(options_val) else []
        record_id = row.get("id", idx)
        mcq_test_records.append({"record_id": record_id, "question": question, "context": context, "options": options})
    mcq_test_df = pd.DataFrame(mcq_test_records)
    print("MCQ test shape:", mcq_test_df.shape)

def predict_mcq_batch(model_save_dir, df, max_choices, max_length, batch_size=4):
    if len(df) == 0:
        return None
    tokenizer = None
    model = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_save_dir)
        model = AutoModelForMultipleChoice.from_pretrained(model_save_dir)
        model.to(DEVICE)
        model.eval()

        def infer_chunk(bs):
            preds_out = []
            for i in range(0, len(df), bs):
                chunk = df.iloc[i:i+bs]
                first_sentences = []
                second_sentences = []
                for _, row in chunk.iterrows():
                    opts = list(row["options"][:max_choices])
                    while len(opts) < max_choices:
                        opts.append("")
                    prompt = f"{row['context']} {row['question']}".strip()
                    first_sentences.extend([prompt] * max_choices)
                    second_sentences.extend(opts)

                encoded = tokenizer(
                    first_sentences, second_sentences, padding=True, truncation=True,
                    max_length=max_length, return_tensors="pt"
                )
                encoded = {k: v.view(len(chunk), max_choices, -1).to(DEVICE) for k, v in encoded.items()}
                with torch.no_grad():
                    with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                        outputs = model(**encoded)
                        logits = outputs.logits
                preds_out.extend(torch.argmax(logits, dim=-1).cpu().numpy().tolist())
            return preds_out

        all_preds, _ = OOM_RUNNER.run(infer_chunk, batch_size)
        return all_preds
    except Exception as e:
        print(f"MCQ test inference failed: {e}")
        traceback.print_exc()
        return None
    finally:
        delete_model_safely(model, tokenizer)

mcq_test_preds = None
if task2_best_mcq_model is not None and len(mcq_test_df) > 0:
    mcq_test_preds = safe_execute(
        predict_mcq_batch, Path(task2_best_mcq_model["save_dir"]), mcq_test_df,
        task2_best_mcq_model["max_choices"], task2_best_mcq_model["max_seq_length"],
        default_return=None, log_prefix="mcq_test_inference"
    )
clear_memory()

MCQ test shape: (500, 4)


In [ ]:
qa_test_raw = task2_raw.get("qa_test")
qa_test_df = pd.DataFrame()

if qa_test_raw is not None:
    qa_test_records = []
    for idx, row in qa_test_raw.iterrows():
        question = clean_text(str(row.get("question"))) if pd.notna(row.get("question")) else ""
        context = clean_text(str(video_transcript_map.get(row.get("video_id"), "")))
        record_id = row.get("id", idx)
        qa_test_records.append({"record_id": record_id, "question": question, "context": context})
    qa_test_df = pd.DataFrame(qa_test_records)
    qa_test_df["answer"] = ""
    print("QA test shape:", qa_test_df.shape)

def generate_qa_test_predictions(model_save_dir, df, max_input_length=384, max_target_length=64, batch_size=8):
    if len(df) == 0:
        return None
    tokenizer = None
    model = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_save_dir)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_save_dir)
        model.to(DEVICE)
        model.eval()

        def infer_chunk(bs):
            preds_out = []
            for i in range(0, len(df), bs):
                chunk = df.iloc[i:i+bs]
                inputs = [f"question: {q} context: {c}" for q, c in zip(chunk["question"], chunk["context"])]
                encoded = tokenizer(inputs, padding=True, truncation=True, max_length=max_input_length, return_tensors="pt")
                encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
                with torch.no_grad():
                    with autocast(dtype=HW.amp_dtype, enabled=HW.use_amp):
                        generated = model.generate(**encoded, max_length=max_target_length, num_beams=4)
                preds_out.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
            return preds_out

        preds, _ = OOM_RUNNER.run(infer_chunk, batch_size)
        return preds
    except Exception as e:
        print(f"QA test inference failed: {e}")
        traceback.print_exc()
        return None
    finally:
        delete_model_safely(model, tokenizer)

qa_test_preds_non_rag = None
qa_test_preds_rag = None

if task2_best_qa_model is not None and len(qa_test_df) > 0:
    qa_test_preds_non_rag = safe_execute(
        generate_qa_test_predictions, Path(task2_best_qa_model["save_dir"]), qa_test_df,
        default_return=None, log_prefix="qa_test_non_rag"
    )

    qa_test_df_rag = qa_test_df.copy()
    if embedding_model is not None and len(video_chunks_df) > 0:
        rag_contexts = []
        for question in qa_test_df["question"]:
            ctx = safe_execute(build_rag_context, question, 3, default_return="", log_prefix="rag_test_context")
            rag_contexts.append(ctx if not is_empty_text(ctx) else "")
        qa_test_df_rag["context"] = rag_contexts

    qa_test_preds_rag = safe_execute(
        generate_qa_test_predictions, Path(task2_best_qa_model["save_dir"]), qa_test_df_rag,
        default_return=None, log_prefix="qa_test_rag"
    )

clear_memory()

QA test shape: (500, 4)


In [ ]:
def write_task2_submission(submission_num, qa_preds, mcq_preds, reddit_preds_df):
    out_dir = SUBMISSION_DIRS_TASK2[submission_num]

    if qa_preds is not None and len(qa_test_df) > 0:
        qa_out = pd.DataFrame({"id": qa_test_df["record_id"], "answer": qa_preds})
        qa_out.to_csv(out_dir / "crypto_test_qna.csv", index=False)
        print(f"Saved {out_dir / 'crypto_test_qna.csv'}")

    if mcq_preds is not None and len(mcq_test_df) > 0:
        mcq_out = pd.DataFrame({"id": mcq_test_df["record_id"], "predicted_answer_idx": mcq_preds})
        mcq_out.to_csv(out_dir / "crypto_test_mcq.csv", index=False)
        print(f"Saved {out_dir / 'crypto_test_mcq.csv'}")

    if reddit_preds_df is not None:
        reddit_preds_df.to_csv(out_dir / "crypto_test_reddit.csv", index=False)
        print(f"Saved {out_dir / 'crypto_test_reddit.csv'}")

reddit_sub_df = reddit_predictions.get("submission_1")

write_task2_submission(1, qa_test_preds_non_rag, mcq_test_preds, reddit_sub_df)

def generate_ensemble_qa_predictions():
    if len(qa_success) < 2 or len(qa_test_df) == 0:
        return qa_test_preds_non_rag
    all_model_preds = []
    for exp in qa_success[:2]:
        preds = safe_execute(
            generate_qa_test_predictions, Path(exp["save_dir"]), qa_test_df,
            default_return=None, log_prefix=f"qa_ensemble_{exp['model_name']}"
        )
        if preds is not None:
            all_model_preds.append(preds)
        clear_memory()
    if len(all_model_preds) < 2:
        return qa_test_preds_non_rag
    final_preds = []
    for i in range(len(qa_test_df)):
        candidates = [p[i] for p in all_model_preds]
        final_preds.append(max(candidates, key=len))
    return final_preds

qa_ensemble_preds = generate_ensemble_qa_predictions()
write_task2_submission(2, qa_ensemble_preds, mcq_test_preds, reddit_sub_df)

final_rag_qa_preds = qa_test_preds_rag if qa_test_preds_rag is not None else qa_test_preds_non_rag
write_task2_submission(3, final_rag_qa_preds, mcq_test_preds, reddit_sub_df)

print("All Task 2 submissions generated")
for i in [1, 2, 3]:
    print(f"Submission_{i}:", os.listdir(SUBMISSION_DIRS_TASK2[i]))

Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_1\crypto_test_qna.csv
Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_1\crypto_test_mcq.csv
Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_1\crypto_test_reddit.csv
Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_2\crypto_test_qna.csv
Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_2\crypto_test_mcq.csv
Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_2\crypto_test_reddit.csv
Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_3\crypto_test_qna.csv
Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_3\crypto_test_mcq.csv
Saved C:\Users\VGST\Desktop\CryptoQA\outputs\Task2\Submission_3\crypto_test_reddit.csv
All Task 2 submissions generated
Submission_1: ['crypto_test_mcq.csv', 'crypto_test_qna.csv', 'crypto_test_reddit.csv']
Submission_2: ['crypto_test_mcq.csv', 'crypto_test_qna.csv', 'crypto_test_reddit.csv']
Submission_3: ['crypto_test_